# AnimalCLEF2026 SAM + YOLO Species Fusion Low-Light Oriented Shape-Aware Submission v20260427

This notebook consumes the SAM+YOLO view factory output and writes upload-ready submission CSVs.

It does **not** preserve any existing submission as the clustering base. Every main candidate starts from singleton seeds and then merges images through shape-aware guarded train-calibrated species-specific pattern edges.

- Lynx: low-light enhanced local spot/texture matching on original-context + SAM/YOLO union views, with left/right orientation guards
- Salamander: SAM-clean local pattern matching on the original image orientation, with head/partial-vs-full-body handling and no rotation/flip preprocessing
- SeaTurtle: conservative independent clean-view scute/head pattern graph
- Texas: original RGB + fused full-canvas SAM/YOLO mask, then full aligned no-oval astro-dot matching

Texas-specific correction in this notebook: Texas horned lizard photos are ventral field views with the head already at the top, so the belly template is **never vertically flipped**.

The notebook also writes `species_fragments/*.csv`. That lets us run future expensive one-species notebooks in parallel and merge their image_id/cluster fragments later without changing the final submission format.

Important Texas detail: this version does **not** use any past submission CSV. Texas is clustered by an independent mutual-rank belly-dot graph, with conservative component-size caps because there are no train identities.

Important graph detail: the previous low-light/oriented run failed because weak single-link edges could percolate into giant components, while the first guarded repair was too easy to over-split. This shape-aware version adds mutual-rank checks, per-image degree caps, realistic Lynx/SeaTurtle edge budgets, stricter Salamander head/full-body bridge rules, species-specific square padding backgrounds, and a species-shape score for choosing top-level `submission.csv`.

The notebook writes only two practical candidates, because the remaining submission budget is tight:

```text
submission_sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_independent_swing.csv
submission_sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_independent_wild.csv
```

The final CSV labels are compacted to `cluster_<SpeciesID>_<number>`, for example `cluster_LynxID2025_33`.

Required Kaggle inputs:

- `animal-clef-2026`
- output dataset from `AnimalCLEF2026_SAM_YOLO_ViewFactory_v20260427`
            

In [ ]:
from pathlib import Path

Path("species_fingerprint_final_swing_v20260426.py").write_text("\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom PIL import Image, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\nBACKGROUND_RGB = np.array([238, 238, 232], dtype=np.uint8)\n\n\ndef read_rgb_with_optional_mask(image_path, mask_path=None, max_side=820):\n    img = Image.open(image_path).convert(\"RGB\")\n    mask_img = None\n    if mask_path:\n        p = Path(mask_path)\n        if p.exists():\n            mask_img = Image.open(p).convert(\"L\")\n            if mask_img.size != img.size:\n                mask_img = mask_img.resize(img.size, Image.Resampling.NEAREST)\n    w, h = img.size\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        size = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))\n        img = img.resize(size, Image.Resampling.BILINEAR)\n        if mask_img is not None:\n            mask_img = mask_img.resize(size, Image.Resampling.NEAREST)\n    rgb = np.asarray(img)\n    mask = None\n    if mask_img is not None:\n        mask = np.where(np.asarray(mask_img) > 127, 255, 0).astype(np.uint8)\n        if float(mask.mean() / 255.0) < 0.01:\n            mask = None\n    return rgb, mask\n\n\ndef estimate_foreground_mask(rgb):\n    diff_bg = np.abs(rgb.astype(np.int16) - BACKGROUND_RGB[None, None, :].astype(np.int16)).sum(axis=2)\n    border = np.concatenate([rgb[:6].reshape(-1, 3), rgb[-6:].reshape(-1, 3), rgb[:, :6].reshape(-1, 3), rgb[:, -6:].reshape(-1, 3)], axis=0)\n    med = np.median(border, axis=0).astype(np.int16)\n    diff_border = np.abs(rgb.astype(np.int16) - med[None, None, :]).sum(axis=2)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    mask = ((diff_bg > 20) & (diff_border > 8)) | ((diff_border > 28) & (hsv[:, :, 1] > 18))\n    mask = mask.astype(np.uint8) * 255\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n_labels > 1:\n        biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n        mask = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.01 or coverage > 0.98:\n        mask = np.full(rgb.shape[:2], 255, dtype=np.uint8)\n    return mask\n\n\ndef bbox_from_mask(mask, pad_frac=0.05):\n    ys, xs = np.where(mask > 0)\n    h, w = mask.shape[:2]\n    if len(xs) == 0:\n        return 0, 0, w, h\n    x0, x1 = int(xs.min()), int(xs.max()) + 1\n    y0, y1 = int(ys.min()), int(ys.max()) + 1\n    pad = int(round(max(x1 - x0, y1 - y0) * pad_frac))\n    return max(0, x0 - pad), max(0, y0 - pad), min(w, x1 + pad), min(h, y1 + pad)\n\n\ndef pca_angle_degrees(mask):\n    ys, xs = np.where(mask > 0)\n    if len(xs) < 8:\n        return 0.0\n    pts = np.stack([xs.astype(np.float32), ys.astype(np.float32)], axis=1)\n    pts -= pts.mean(axis=0, keepdims=True)\n    _, _, vt = np.linalg.svd(pts, full_matrices=False)\n    vx, vy = vt[0]\n    return float(np.degrees(np.arctan2(vy, vx)))\n\n\ndef rotate_bound(rgb, mask, angle):\n    h, w = rgb.shape[:2]\n    center = (w / 2.0, h / 2.0)\n    m = cv2.getRotationMatrix2D(center, angle, 1.0)\n    cos = abs(m[0, 0])\n    sin = abs(m[0, 1])\n    nw = int((h * sin) + (w * cos))\n    nh = int((h * cos) + (w * sin))\n    m[0, 2] += (nw / 2.0) - center[0]\n    m[1, 2] += (nh / 2.0) - center[1]\n    rr = cv2.warpAffine(rgb, m, (nw, nh), flags=cv2.INTER_LINEAR, borderValue=tuple(int(x) for x in BACKGROUND_RGB))\n    mm = cv2.warpAffine(mask, m, (nw, nh), flags=cv2.INTER_NEAREST, borderValue=0)\n    return rr, np.where(mm > 0, 255, 0).astype(np.uint8)\n", encoding="utf-8")
Path("texas_astrodot_2025reuse_v20260426.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nTHIS_DIR = Path(__file__).resolve().parent\nif str(THIS_DIR) not in sys.path:\n    sys.path.insert(0, str(THIS_DIR))\n\nimport species_fingerprint_final_swing_v20260426 as core\n\n\nVERSION = \"texas_astrodot_2025reuse_v20260426\"\nSEED = 20260426\nTEXAS = \"TexasHornedLizards\"\nREUSED_SPECIES = [\"LynxID2025\", \"SalamanderID2025\", \"SeaTurtleID2022\"]\nBACKGROUND = np.array([238, 238, 232], dtype=np.uint8)\n\n\n@dataclass\nclass TexasDotItem:\n    image_id: int\n    current_cluster: str\n    source_path: str\n    view_path: str\n    view_source: str\n    belly_rgb: np.ndarray\n    belly_mask: np.ndarray\n    dot_heat: np.ndarray\n    dot_points: np.ndarray\n    vector: np.ndarray\n    quality: float\n    debug: dict\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> int:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return ra\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return ra\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Texas astro-dot stacking branch plus 2025-winner-style train \"\n            \"local-match validation for reused AnimalCLEF species. This script \"\n            \"writes upload-ready CSVs only; it never submits to Kaggle.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--current-best-submission\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--max-side\", type=int, default=760)\n    parser.add_argument(\"--texas-canvas-w\", type=int, default=224)\n    parser.add_argument(\"--texas-canvas-h\", type=int, default=320)\n    parser.add_argument(\"--texas-pair-budget\", type=int, default=12000, help=\"0 means all Texas pairs.\")\n    parser.add_argument(\"--reused-train-pair-budget\", type=int, default=1800)\n    parser.add_argument(\"--reused-max-train-images\", type=int, default=650)\n    parser.add_argument(\"--reused-max-per-identity\", type=int, default=8)\n    parser.add_argument(\"--skip-reused-validation\", action=\"store_true\")\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--visual-limit\", type=int, default=18)\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n            Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\animal-clef-2026\"),\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_sam_manifest(user_value: str | None) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/reports/view_manifest_sam3_all_species.csv\"),\n        Path.cwd() / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n        Path.cwd().parent / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent]:\n        if not base.exists():\n            continue\n        try:\n            matches = list(base.rglob(\"view_manifest_sam3_all_species.csv\"))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda x: len(str(x)))\n            return matches[0].resolve()\n    return None\n\n\ndef export_root_from_manifest(manifest_path: Path) -> Path:\n    if manifest_path.parent.name == \"reports\":\n        return manifest_path.parent.parent\n    return manifest_path.parent\n\n\ndef remap_export_path(path_value: object, export_root: Path | None) -> Path | None:\n    if path_value is None or pd.isna(path_value):\n        return None\n    s = str(path_value).strip()\n    if not s:\n        return None\n    p = Path(s)\n    if p.exists():\n        return p.resolve()\n    if export_root is None:\n        return None\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\n        \"animalclef_sam3_views_cache/\",\n        \"views/\",\n        \"mask_loose_square/\",\n        \"mask_full_square/\",\n    ]\n    for marker in markers:\n        if marker not in normalized:\n            continue\n        rel = normalized.split(marker, 1)[1]\n        if marker != \"animalclef_sam3_views_cache/\":\n            rel = marker + rel\n        candidate = export_root / Path(rel)\n        if candidate.exists():\n            return candidate.resolve()\n    return None\n\n\ndef prepare_metadata(data_root: Path, sam_manifest: Path | None) -> tuple[pd.DataFrame, dict]:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").reset_index(drop=True)\n    if \"row_idx\" not in metadata.columns:\n        metadata[\"row_idx\"] = np.arange(len(metadata), dtype=np.int64)\n    if \"species_id\" not in metadata.columns:\n        metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(metadata[\"path\"].str.contains(\"/test/\"), \"test\", \"train\")\n    if \"orientation\" not in metadata.columns:\n        metadata[\"orientation\"] = \"unknown\"\n    if \"identity\" not in metadata.columns:\n        metadata[\"identity\"] = \"\"\n    metadata[\"source_path\"] = metadata[\"path\"].map(lambda p: str(data_root / str(p)))\n    metadata[\"view_path\"] = metadata[\"source_path\"].astype(str)\n    metadata[\"view_source\"] = \"original_fallback\"\n    metadata[\"sam_view_path\"] = \"\"\n    metadata[\"mask_path\"] = \"\"\n    metadata[\"mask_ok\"] = False\n\n    info = {\n        \"manifest_path\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_rows\": 0,\n        \"resolved_sam_views\": 0,\n        \"resolved_masks\": 0,\n    }\n    if sam_manifest is None:\n        return metadata, info\n\n    manifest = pd.read_csv(sam_manifest)\n    info[\"manifest_rows\"] = int(len(manifest))\n    export_root = export_root_from_manifest(sam_manifest)\n    merge_key = \"row_idx\" if \"row_idx\" in manifest.columns else \"image_id\" if \"image_id\" in manifest.columns else None\n    if merge_key is None:\n        return metadata, info\n    merged = metadata.merge(manifest, on=merge_key, how=\"left\", suffixes=(\"\", \"_sam\"))\n\n    view_paths: list[str] = []\n    sam_view_paths: list[str] = []\n    mask_paths: list[str] = []\n    view_sources: list[str] = []\n    mask_ok: list[bool] = []\n    sam_count = 0\n    mask_count = 0\n    for row in merged.to_dict(\"records\"):\n        resolved_sam_view = None\n        for col in [\"loose_path\", \"mask_loose_path\", \"mask_full_path\", \"view_path\"]:\n            if col in row:\n                resolved_sam_view = remap_export_path(row.get(col), export_root)\n                if resolved_sam_view is not None:\n                    break\n        resolved_mask = None\n        for col in [\"mask_path\", \"binary_mask_path\"]:\n            if col in row:\n                resolved_mask = remap_export_path(row.get(col), export_root)\n                if resolved_mask is not None:\n                    break\n        if resolved_sam_view is not None:\n            view_paths.append(str(resolved_sam_view))\n            sam_view_paths.append(str(resolved_sam_view))\n            view_sources.append(\"sam_clean_primary\")\n            sam_count += 1\n        else:\n            view_paths.append(str(row[\"source_path\"]))\n            sam_view_paths.append(\"\")\n            view_sources.append(\"original_fallback\")\n        if resolved_mask is not None:\n            mask_paths.append(str(resolved_mask))\n            mask_ok.append(True)\n            mask_count += 1\n        else:\n            mask_paths.append(\"\")\n            mask_ok.append(False)\n    merged[\"view_path\"] = view_paths\n    merged[\"sam_view_path\"] = sam_view_paths\n    merged[\"view_source\"] = view_sources\n    merged[\"mask_path\"] = mask_paths\n    merged[\"mask_ok\"] = mask_ok\n    info[\"resolved_sam_views\"] = int(sam_count)\n    info[\"resolved_masks\"] = int(mask_count)\n    return merged, info\n\n\ndef find_current_best(user_value: str | None, data_root: Path) -> Path:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    filename = core.CURRENT_BEST_FILENAME\n    roots = [\n        Path(\"/kaggle/input\"),\n        Path(\"/kaggle/working\"),\n        Path.cwd(),\n        Path.cwd().parent,\n        data_root.parent,\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\"),\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\kaggle_upload_atlas_inputs_v20260426\"),\n    ]\n    found = core.find_file_everywhere(filename, roots)\n    if found is None:\n        raise FileNotFoundError(f\"Could not find {filename}.\")\n    return found\n\n\ndef normalize(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32, copy=False)\n    norm = float(np.linalg.norm(vec))\n    if norm > 0:\n        vec = vec / norm\n    return vec.astype(np.float32, copy=False)\n\n\ndef clean_mask(mask: np.ndarray, shape: tuple[int, int] | None = None) -> np.ndarray:\n    m = np.where(mask > 0, 255, 0).astype(np.uint8)\n    if shape is not None and m.shape[:2] != shape:\n        m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)\n        m = np.where(m > 0, 255, 0).astype(np.uint8)\n    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))\n    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k1)\n    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k2)\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, 8)\n    if n > 1:\n        areas = stats[1:, cv2.CC_STAT_AREA]\n        if len(areas):\n            biggest = 1 + int(np.argmax(areas))\n            m = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    if float(m.mean() / 255.0) < 0.01:\n        m = np.full(m.shape[:2], 255, dtype=np.uint8)\n    return m\n\n\ndef read_rgb_mask(row: dict, max_side: int) -> tuple[np.ndarray, np.ndarray, float]:\n    rgb, mask = core.read_rgb_with_optional_mask(row.get(\"view_path\", row.get(\"source_path\")), row.get(\"mask_path\"), max_side)\n    quality = 1.0\n    if mask is None:\n        mask = core.estimate_foreground_mask(rgb)\n        quality = 0.86\n    mask = clean_mask(mask, rgb.shape[:2])\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.015 or coverage > 0.98:\n        mask = core.estimate_foreground_mask(rgb)\n        mask = clean_mask(mask, rgb.shape[:2])\n        quality *= 0.78\n    return rgb, mask, quality\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad: float = 0.08) -> tuple[np.ndarray, np.ndarray]:\n    x1, y1, x2, y2 = core.bbox_from_mask(mask, pad)\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef align_vertical(rgb: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, 0.08)\n    angle = core.pca_angle_degrees(crop_mask)\n    rotate_angle = 90.0 - angle\n    if abs(rotate_angle) > 1.5:\n        crop_rgb, crop_mask = core.rotate_bound(crop_rgb, crop_mask, rotate_angle)\n        crop_rgb, crop_mask = crop_to_mask(crop_rgb, crop_mask, 0.04)\n    h, w = crop_mask.shape[:2]\n    widths = []\n    for yf in [0.18, 0.30, 0.70, 0.84]:\n        y = int(np.clip(round(h * yf), 0, h - 1))\n        xs = np.where(crop_mask[y] > 0)[0]\n        widths.append(float(xs.max() - xs.min() + 1) if len(xs) else 0.0)\n    top_width = max(widths[:2])\n    bottom_width = max(widths[2:])\n    flipped = False\n    # Most Texas belly photos have the head/shoulder end above the tail end.\n    # If the bottom bands are clearly wider, canonicalize by vertical flip.\n    if bottom_width > top_width * 1.18:\n        crop_rgb = crop_rgb[::-1, :, :].copy()\n        crop_mask = crop_mask[::-1, :].copy()\n        flipped = True\n    return crop_rgb, crop_mask, {\n        \"pca_angle\": float(angle),\n        \"rotate_angle\": float(rotate_angle),\n        \"top_width\": float(top_width),\n        \"bottom_width\": float(bottom_width),\n        \"flipped_vertical\": bool(flipped),\n    }\n\n\ndef clahe_u8(channel: np.ndarray) -> np.ndarray:\n    channel = np.clip(channel, 0, 255).astype(np.uint8)\n    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(channel)\n\n\ndef texas_dot_heat(belly_rgb: np.ndarray, belly_mask: np.ndarray) -> np.ndarray:\n    lab = cv2.cvtColor(belly_rgb, cv2.COLOR_RGB2LAB)\n    l_eq = clahe_u8(lab[:, :, 0])\n    dark = (255.0 - l_eq.astype(np.float32)) / 255.0\n    blackhat = cv2.morphologyEx(l_eq, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13)))\n    spot = 0.58 * (blackhat.astype(np.float32) / 255.0) + 0.42 * dark\n    valid = belly_mask > 0\n    if valid.sum() > 20:\n        vals = spot[valid]\n        lo = float(np.percentile(vals, 12))\n        hi = float(np.percentile(vals, 98))\n        spot = (spot - lo) / max(1e-6, hi - lo)\n    spot = np.clip(spot, 0, 1)\n    spot[~valid] = 0.0\n    spot = cv2.GaussianBlur(spot, (3, 3), 0)\n    return spot.astype(np.float32)\n\n\ndef texas_belly_template(row: dict, current_cluster: str, args: argparse.Namespace) -> TexasDotItem:\n    rgb, mask, quality = read_rgb_mask(row, args.max_side)\n    aligned_rgb, aligned_mask, debug = align_vertical(rgb, mask)\n    w = int(args.texas_canvas_w)\n    h = int(args.texas_canvas_h)\n    belly_rgb = cv2.resize(aligned_rgb, (w, h), interpolation=cv2.INTER_AREA)\n    belly_mask = cv2.resize(aligned_mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    belly_mask = clean_mask(belly_mask)\n    ellipse = np.zeros((h, w), dtype=np.uint8)\n    cv2.ellipse(\n        ellipse,\n        (w // 2, int(h * 0.49)),\n        (int(w * 0.38), int(h * 0.35)),\n        0,\n        0,\n        360,\n        255,\n        -1,\n    )\n    belly_mask = cv2.bitwise_and(belly_mask, ellipse)\n    if float(belly_mask.mean() / 255.0) < 0.035:\n        belly_mask = ellipse\n        quality *= 0.72\n    heat = texas_dot_heat(belly_rgb, belly_mask)\n    points = dot_points_from_heat(heat, belly_mask)\n    small = cv2.resize(heat, (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    mask_small = cv2.resize((belly_mask > 0).astype(np.float32), (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    vector = normalize(np.concatenate([small * mask_small, mask_small * 0.18]).astype(np.float32))\n    quality *= min(1.0, 0.72 + 0.28 * min(1.0, len(points) / 35.0))\n    return TexasDotItem(\n        image_id=int(row[\"image_id\"]),\n        current_cluster=str(current_cluster),\n        source_path=str(row.get(\"source_path\", \"\")),\n        view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n        view_source=str(row.get(\"view_source\", \"\")),\n        belly_rgb=belly_rgb,\n        belly_mask=belly_mask,\n        dot_heat=heat,\n        dot_points=points,\n        vector=vector,\n        quality=float(quality),\n        debug=debug,\n    )\n\n\ndef dot_points_from_heat(heat: np.ndarray, mask: np.ndarray) -> np.ndarray:\n    valid = mask > 0\n    if int(valid.sum()) < 40:\n        return np.zeros((0, 4), dtype=np.float32)\n    vals = heat[valid]\n    thr = max(float(np.percentile(vals, 86)), float(vals.mean() + 0.60 * vals.std()))\n    binary = np.zeros(heat.shape, dtype=np.uint8)\n    binary[(heat >= thr) & valid] = 255\n    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, 8)\n    h, w = heat.shape[:2]\n    total = float(h * w)\n    pts: list[list[float]] = []\n    min_area = max(2.0, total * 0.00005)\n    max_area = total * 0.012\n    for idx in range(1, n):\n        area = float(stats[idx, cv2.CC_STAT_AREA])\n        if area < min_area or area > max_area:\n            continue\n        bw = max(1.0, float(stats[idx, cv2.CC_STAT_WIDTH]))\n        bh = max(1.0, float(stats[idx, cv2.CC_STAT_HEIGHT]))\n        if max(bw / bh, bh / bw) > 5.5:\n            continue\n        cx, cy = centroids[idx]\n        strength = float(heat[labels == idx].mean())\n        pts.append([float(cx) / max(1, w - 1), float(cy) / max(1, h - 1), area / total, strength])\n    if not pts:\n        return np.zeros((0, 4), dtype=np.float32)\n    pts_arr = np.asarray(pts, dtype=np.float32)\n    order = np.argsort(-pts_arr[:, 3])\n    return pts_arr[order[:260]]\n\n\ndef shifted_slices(shape: tuple[int, int], dx: int, dy: int):\n    h, w = shape\n    xa1 = max(0, dx)\n    xb1 = max(0, -dx)\n    ya1 = max(0, dy)\n    yb1 = max(0, -dy)\n    ww = w - abs(dx)\n    hh = h - abs(dy)\n    if ww <= 8 or hh <= 8:\n        return None\n    return (slice(ya1, ya1 + hh), slice(xa1, xa1 + ww)), (slice(yb1, yb1 + hh), slice(xb1, xb1 + ww))\n\n\ndef dot_map_sharpness(heat: np.ndarray, mask: np.ndarray) -> float:\n    valid = mask > 0\n    if int(valid.sum()) < 20:\n        return 0.0\n    vals = heat[valid].astype(np.float32)\n    p95 = float(np.percentile(vals, 95))\n    p50 = float(np.percentile(vals, 50))\n    lap = cv2.Laplacian(heat, cv2.CV_32F, ksize=3)\n    lap_energy = float(np.mean(np.abs(lap[valid])))\n    return float(max(0.0, p95 - p50) + 0.35 * lap_energy)\n\n\ndef masked_corr_and_stack(\n    a_heat: np.ndarray,\n    a_mask: np.ndarray,\n    b_heat: np.ndarray,\n    b_mask: np.ndarray,\n    dx: int,\n    dy: int,\n) -> tuple[float, float, float]:\n    slices = shifted_slices(a_mask.shape[:2], dx, dy)\n    if slices is None:\n        return 0.0, 0.0, 0.0\n    sa, sb = slices\n    ma = a_mask[sa] > 0\n    mb = b_mask[sb] > 0\n    overlap_mask = ma & mb\n    overlap = float(overlap_mask.mean()) if overlap_mask.size else 0.0\n    if overlap < 0.04 or int(overlap_mask.sum()) < 40:\n        return 0.0, overlap, 0.0\n    va = a_heat[sa][overlap_mask].astype(np.float32)\n    vb = b_heat[sb][overlap_mask].astype(np.float32)\n    am = va - float(va.mean())\n    bm = vb - float(vb.mean())\n    denom = float(np.linalg.norm(am) * np.linalg.norm(bm))\n    corr = float(np.dot(am, bm) / denom) if denom > 1e-6 else 0.0\n    corr01 = float(np.clip((corr + 1.0) * 0.5, 0.0, 1.0))\n    stack = np.zeros_like(a_heat, dtype=np.float32)\n    stack_mask = np.zeros_like(a_mask, dtype=np.uint8)\n    stack[sa] = 0.5 * (a_heat[sa] + b_heat[sb])\n    stack_mask[sa] = np.where(overlap_mask, 255, 0).astype(np.uint8)\n    sharp_stack = dot_map_sharpness(stack, stack_mask)\n    sharp_a = dot_map_sharpness(a_heat[sa], np.where(overlap_mask, 255, 0).astype(np.uint8))\n    sharp_b = dot_map_sharpness(b_heat[sb], np.where(overlap_mask, 255, 0).astype(np.uint8))\n    baseline = 0.5 * (sharp_a + sharp_b)\n    # Same individuals should retain or improve peakiness after stacking;\n    # different dot fields smear and reduce the normalized sharpness.\n    stack_gain = float(np.clip(sharp_stack / max(1e-6, baseline), 0.0, 1.35) / 1.35)\n    return corr01, overlap, stack_gain\n\n\ndef chamfer_dot_score(points_a: np.ndarray, points_b: np.ndarray, dx_norm: float, dy_norm: float) -> float:\n    if len(points_a) < 5 or len(points_b) < 5:\n        return 0.0\n    a = points_a[:, :2].astype(np.float32)\n    b = points_b[:, :2].astype(np.float32).copy()\n    b[:, 0] += dx_norm\n    b[:, 1] += dy_norm\n    diff = a[:, None, :] - b[None, :, :]\n    dist = np.sqrt(np.maximum(0.0, (diff * diff).sum(axis=2)))\n    da = dist.min(axis=1)\n    db = dist.min(axis=0)\n    ka = np.argsort(da)[: min(len(da), 120)]\n    kb = np.argsort(db)[: min(len(db), 120)]\n    mean_d = 0.5 * (float(da[ka].mean()) + float(db[kb].mean()))\n    return float(np.exp(-mean_d / 0.050))\n\n\ndef phase_shift(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> tuple[int, int]:\n    try:\n        aw = (a * mask).astype(np.float32)\n        bw = (b * mask).astype(np.float32)\n        shift, _ = cv2.phaseCorrelate(aw, bw)\n        dx = int(np.clip(round(shift[0]), -18, 18))\n        dy = int(np.clip(round(shift[1]), -18, 18))\n        return dx, dy\n    except Exception:\n        return 0, 0\n\n\ndef transform_heat_mask_points(item: TexasDotItem, transform: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    heat = item.dot_heat\n    mask = item.belly_mask\n    pts = item.dot_points.copy()\n    if transform in {\"flip_x\", \"flip_xy\"}:\n        heat = heat[:, ::-1].copy()\n        mask = mask[:, ::-1].copy()\n        if len(pts):\n            pts[:, 0] = 1.0 - pts[:, 0]\n    if transform in {\"flip_y\", \"flip_xy\"}:\n        heat = heat[::-1, :].copy()\n        mask = mask[::-1, :].copy()\n        if len(pts):\n            pts[:, 1] = 1.0 - pts[:, 1]\n    return heat, mask, pts\n\n\ndef texas_pair_score(a: TexasDotItem, b: TexasDotItem) -> dict:\n    desc_cos = float(np.dot(a.vector, b.vector))\n    h, w = a.dot_heat.shape[:2]\n    best = {\n        \"score\": 0.0,\n        \"corr\": 0.0,\n        \"overlap\": 0.0,\n        \"stack_gain\": 0.0,\n        \"point_score\": 0.0,\n        \"descriptor_cosine\": desc_cos,\n        \"dx\": 0,\n        \"dy\": 0,\n        \"transform\": \"identity\",\n    }\n    transforms = [\"identity\", \"flip_y\"]\n    common_mask = np.where((a.belly_mask > 0), 1.0, 0.0).astype(np.float32)\n    for transform in transforms:\n        b_heat, b_mask, b_pts = transform_heat_mask_points(b, transform)\n        base_dx, base_dy = phase_shift(a.dot_heat, b_heat, common_mask)\n        candidates = {(0, 0), (base_dx, base_dy)}\n        for dx0, dy0 in [(base_dx, base_dy), (0, 0)]:\n            for ddx in (-8, 0, 8):\n                for ddy in (-8, 0, 8):\n                    candidates.add((int(np.clip(dx0 + ddx, -24, 24)), int(np.clip(dy0 + ddy, -24, 24))))\n        for dx, dy in candidates:\n            corr, overlap, stack_gain = masked_corr_and_stack(a.dot_heat, a.belly_mask, b_heat, b_mask, dx, dy)\n            if overlap < 0.045:\n                continue\n            point_score = chamfer_dot_score(a.dot_points, b_pts, dx / max(1, w - 1), dy / max(1, h - 1))\n            fused = (\n                0.34 * corr\n                + 0.31 * point_score\n                + 0.23 * stack_gain\n                + 0.12 * max(0.0, desc_cos)\n            )\n            if transform != \"identity\":\n                fused *= 0.92\n            fused *= min(a.quality, b.quality)\n            if fused > best[\"score\"]:\n                best = {\n                    \"score\": float(fused),\n                    \"corr\": float(corr),\n                    \"overlap\": float(overlap),\n                    \"stack_gain\": float(stack_gain),\n                    \"point_score\": float(point_score),\n                    \"descriptor_cosine\": desc_cos,\n                    \"dx\": int(dx),\n                    \"dy\": int(dy),\n                    \"transform\": transform,\n                }\n    best[\"points_a\"] = int(len(a.dot_points))\n    best[\"points_b\"] = int(len(b.dot_points))\n    best[\"quality_a\"] = float(a.quality)\n    best[\"quality_b\"] = float(b.quality)\n    best[\"same_current_cluster\"] = bool(a.current_cluster == b.current_cluster)\n    return best\n\n\ndef score_all_texas_pairs(items: list[TexasDotItem], pair_budget: int = 0) -> pd.DataFrame:\n    vectors = np.stack([it.vector for it in items]).astype(np.float32)\n    sim = vectors @ vectors.T\n    ids = [it.image_id for it in items]\n    by_id = {it.image_id: it for it in items}\n    pairs = [(ids[i], ids[j]) for i in range(len(ids)) for j in range(i + 1, len(ids))]\n    if pair_budget and len(pairs) > pair_budget:\n        id_to_idx = {image_id: idx for idx, image_id in enumerate(ids)}\n        pairs.sort(key=lambda p: float(sim[id_to_idx[p[0]], id_to_idx[p[1]]]), reverse=True)\n        current_pairs = [p for p in pairs if by_id[p[0]].current_cluster == by_id[p[1]].current_cluster]\n        selected = current_pairs + [p for p in pairs if by_id[p[0]].current_cluster != by_id[p[1]].current_cluster]\n        pairs = selected[:pair_budget]\n    rows = []\n    for idx, (a_id, b_id) in enumerate(pairs, start=1):\n        a = by_id[int(a_id)]\n        b = by_id[int(b_id)]\n        score = texas_pair_score(a, b)\n        rows.append(\n            {\n                \"species\": TEXAS,\n                \"image_id_a\": int(a_id),\n                \"image_id_b\": int(b_id),\n                \"current_cluster_a\": a.current_cluster,\n                \"current_cluster_b\": b.current_cluster,\n                **score,\n            }\n        )\n        if idx % 5000 == 0:\n            print(f\"[Texas astro-dot] scored {idx}/{len(pairs)} pairs\")\n    return pd.DataFrame(rows)\n\n\ndef relabel_components(ids: list[int], uf: UnionFind, variant: str) -> dict[int, str]:\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(ids):\n        comp = uf.find(image_id)\n        if comp not in comp_order:\n            comp_order[comp] = len(comp_order)\n        labels[image_id] = f\"cluster_TexasHornedLizards_astrodot_{variant}_{comp_order[comp]}\"\n    return labels\n\n\ndef texas_variant_labels(items: list[TexasDotItem], pair_scores: pd.DataFrame, variant: str) -> dict[int, str]:\n    ids = [it.image_id for it in items]\n    by_cluster: dict[str, list[int]] = {}\n    current_by_id = {it.image_id: it.current_cluster for it in items}\n    for it in items:\n        by_cluster.setdefault(it.current_cluster, []).append(it.image_id)\n\n    # Thresholds are deliberately strict. Texas has no train labels, so this\n    # branch should produce surgical candidates, not a free-running clusterer.\n    if variant == \"split_strict\":\n        keep_thr, merge_thr, merge_rank = 0.430, 9.9, 0\n    elif variant == \"merge_ultra\":\n        keep_thr, merge_thr, merge_rank = 0.0, 0.650, 2\n    elif variant == \"splitmerge_guarded\":\n        keep_thr, merge_thr, merge_rank = 0.405, 0.620, 2\n    else:\n        keep_thr, merge_thr, merge_rank = 0.385, 0.590, 3\n\n    uf = UnionFind(ids)\n    if variant in {\"merge_ultra\"}:\n        # Preserve current clusters, then add only extremely high-confidence\n        # cross-cluster dot-stack merges.\n        for members in by_cluster.values():\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n    else:\n        # Split current clusters by verified intra-cluster dot support.\n        for cluster, members in by_cluster.items():\n            if len(members) <= 1:\n                continue\n            g = pair_scores[\n                pair_scores[\"image_id_a\"].isin(members)\n                & pair_scores[\"image_id_b\"].isin(members)\n                & pair_scores[\"same_current_cluster\"].astype(bool)\n            ]\n            for row in g[g[\"score\"].astype(float) >= keep_thr].itertuples(index=False):\n                uf.union(int(row.image_id_a), int(row.image_id_b))\n\n    if merge_rank > 0:\n        cross = pair_scores[~pair_scores[\"same_current_cluster\"].astype(bool)].copy()\n        if not cross.empty:\n            cross = cross[\n                (cross[\"score\"].astype(float) >= merge_thr)\n                & (cross[\"overlap\"].astype(float) >= 0.13)\n                & (cross[\"point_score\"].astype(float) >= 0.36)\n                & (cross[\"stack_gain\"].astype(float) >= 0.43)\n            ].copy()\n            if not cross.empty:\n                neighbors: dict[int, list[tuple[int, float]]] = {i: [] for i in ids}\n                for row in cross.itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    s = float(row.score)\n                    neighbors[a].append((b, s))\n                    neighbors[b].append((a, s))\n                ranks: dict[tuple[int, int], int] = {}\n                for node, vals in neighbors.items():\n                    vals.sort(key=lambda x: -x[1])\n                    for rank, (other, _) in enumerate(vals, start=1):\n                        ranks[(node, other)] = rank\n                for row in cross.sort_values(\"score\", ascending=False).itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    if current_by_id[a] == current_by_id[b]:\n                        continue\n                    if ranks.get((a, b), 999) <= merge_rank and ranks.get((b, a), 999) <= merge_rank:\n                        uf.union(a, b)\n    components: dict[int, list[int]] = {}\n    for image_id in ids:\n        components.setdefault(uf.find(image_id), []).append(image_id)\n    current_sets = {cluster: set(members) for cluster, members in by_cluster.items()}\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for root, members in sorted(components.items(), key=lambda kv: min(kv[1])):\n        member_set = set(members)\n        member_current = {current_by_id[i] for i in members}\n        if len(member_current) == 1:\n            current_cluster = next(iter(member_current))\n            if member_set == current_sets.get(current_cluster, set()):\n                for image_id in members:\n                    labels[image_id] = current_cluster\n                continue\n        comp_order[root] = len(comp_order)\n        new_label = f\"cluster_TexasHornedLizards_astrodot_{variant}_{comp_order[root]}\"\n        for image_id in members:\n            labels[image_id] = new_label\n    return labels\n\n\ndef summarize_texas_variant(labels: dict[int, str], pair_scores: pd.DataFrame, variant: str, current_labels: dict[int, str]) -> dict:\n    counts = pd.Series(list(labels.values())).value_counts()\n    changed = int(sum(1 for i, lab in labels.items() if lab != current_labels.get(i)))\n    return {\n        \"variant\": variant,\n        \"species\": TEXAS,\n        \"n_images\": int(len(labels)),\n        \"n_clusters\": int(counts.shape[0]),\n        \"singletons\": int((counts == 1).sum()),\n        \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n        \"rows_changed_vs_current\": changed,\n        \"pair_score_mean\": float(pair_scores[\"score\"].mean()) if not pair_scores.empty else 0.0,\n        \"pair_score_p95\": float(pair_scores[\"score\"].quantile(0.95)) if not pair_scores.empty else 0.0,\n        \"same_current_mean\": float(pair_scores.loc[pair_scores[\"same_current_cluster\"].astype(bool), \"score\"].mean())\n        if not pair_scores.empty and pair_scores[\"same_current_cluster\"].astype(bool).any()\n        else 0.0,\n        \"cross_current_p99\": float(pair_scores.loc[~pair_scores[\"same_current_cluster\"].astype(bool), \"score\"].quantile(0.99))\n        if not pair_scores.empty and (~pair_scores[\"same_current_cluster\"].astype(bool)).any()\n        else 0.0,\n    }\n\n\ndef root_sift(desc: np.ndarray | None) -> np.ndarray | None:\n    if desc is None or len(desc) == 0:\n        return desc\n    desc = desc.astype(np.float32)\n    desc /= np.maximum(1e-7, desc.sum(axis=1, keepdims=True))\n    return np.sqrt(desc).astype(np.float32)\n\n\ndef extract_local_features(row: dict, species: str, max_side: int, detector) -> tuple[list, np.ndarray | None, np.ndarray]:\n    cfg = core.SPECIES_CONFIGS[species]\n    rgb, mask = core.read_rgb_with_optional_mask(row.get(\"view_path\", row.get(\"source_path\")), row.get(\"mask_path\"), max_side)\n    if mask is None:\n        mask = core.estimate_foreground_mask(rgb)\n    roi_rgb, roi_mask, _ = core.species_roi(\n        rgb,\n        species,\n        str(row.get(\"orientation\", \"unknown\")).lower(),\n        cfg,\n        mask_override=mask,\n    )\n    gray = core.pattern_gray(roi_rgb, species)\n    kps, desc = detector.detectAndCompute(gray, roi_mask)\n    if kps is None:\n        kps = []\n    desc = root_sift(desc)\n    return kps, desc, roi_mask\n\n\ndef local_pair_score(a_feat, b_feat, species: str) -> dict:\n    kps_a, desc_a, mask_a = a_feat\n    kps_b, desc_b, mask_b = b_feat\n    if desc_a is None or desc_b is None or len(desc_a) < 4 or len(desc_b) < 4:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": 0, \"inlier_ratio\": 0.0}\n    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)\n    try:\n        knn = matcher.knnMatch(desc_a, desc_b, k=2)\n    except Exception:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": 0, \"inlier_ratio\": 0.0}\n    ratio = 0.80 if species == \"SalamanderID2025\" else 0.76\n    good = []\n    for pair in knn:\n        if len(pair) < 2:\n            continue\n        m, n = pair\n        if m.distance < ratio * n.distance:\n            good.append(m)\n    if len(good) < 4:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": len(good), \"inlier_ratio\": 0.0}\n    pts_a = np.float32([kps_a[m.queryIdx].pt for m in good]).reshape(-1, 2)\n    pts_b = np.float32([kps_b[m.trainIdx].pt for m in good]).reshape(-1, 2)\n    _, inlier_mask = cv2.estimateAffinePartial2D(\n        pts_a,\n        pts_b,\n        method=cv2.RANSAC,\n        ransacReprojThreshold=5.0,\n        maxIters=1600,\n        confidence=0.995,\n    )\n    inliers = int(inlier_mask.reshape(-1).sum()) if inlier_mask is not None else 0\n    denom = max(1, min(len(kps_a), len(kps_b), len(good)))\n    inlier_ratio = float(inliers / denom)\n    inlier_term = min(1.0, inliers / (28.0 if species != \"SalamanderID2025\" else 18.0))\n    ratio_term = min(1.0, inlier_ratio / 0.25)\n    match_term = min(1.0, len(good) / 60.0)\n    score = 0.55 * inlier_term + 0.30 * ratio_term + 0.15 * match_term\n    return {\"score\": float(score), \"inliers\": inliers, \"good_matches\": int(len(good)), \"inlier_ratio\": inlier_ratio}\n\n\ndef sample_train_rows(rows: pd.DataFrame, max_images: int, max_per_identity: int) -> pd.DataFrame:\n    labeled = rows[rows[\"identity\"].fillna(\"\").astype(str).str.len().gt(0)].copy()\n    if labeled.empty:\n        return labeled\n    rng = random.Random(SEED)\n    chosen = []\n    for _, group in labeled.groupby(\"identity\"):\n        recs = group.sort_values(\"image_id\").to_dict(\"records\")\n        if len(recs) > max_per_identity:\n            recs = rng.sample(recs, max_per_identity)\n        chosen.extend(recs)\n    if max_images and len(chosen) > max_images:\n        chosen = rng.sample(chosen, max_images)\n    chosen.sort(key=lambda r: int(r[\"image_id\"]))\n    return pd.DataFrame(chosen)\n\n\ndef sample_validation_pairs(items: list[dict], budget: int) -> list[tuple[int, int]]:\n    rng = random.Random(SEED)\n    ids = [int(r[\"image_id\"]) for r in items]\n    identities = {int(r[\"image_id\"]): str(r.get(\"identity\", \"\")) for r in items}\n    by_identity: dict[str, list[int]] = {}\n    for i in ids:\n        by_identity.setdefault(identities[i], []).append(i)\n    positives = []\n    for members in by_identity.values():\n        if len(members) < 2:\n            continue\n        combos = list(itertools.combinations(sorted(members), 2))\n        if len(combos) > 10:\n            combos = rng.sample(combos, 10)\n        positives.extend(combos)\n    if len(positives) > budget // 2:\n        positives = rng.sample(positives, max(1, budget // 2))\n    negatives = set()\n    target_neg = min(budget - len(positives), max(600 if positives else budget, len(positives) * 2))\n    attempts = 0\n    while len(negatives) < target_neg and attempts < max(100, target_neg * 35) and len(ids) >= 2:\n        a, b = rng.sample(ids, 2)\n        attempts += 1\n        if identities[a] == identities[b]:\n            continue\n        negatives.add((a, b) if a < b else (b, a))\n    pairs = positives + sorted(negatives)\n    if len(pairs) > budget:\n        pairs = rng.sample(pairs, budget)\n    return [(int(a), int(b)) for a, b in pairs]\n\n\ndef auc_rank(y_true: np.ndarray, scores: np.ndarray) -> float:\n    y = y_true.astype(bool)\n    n_pos = int(y.sum())\n    n_neg = int((~y).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float(\"nan\")\n    order = np.argsort(scores)\n    ranks = np.empty_like(order, dtype=np.float64)\n    ranks[order] = np.arange(1, len(scores) + 1, dtype=np.float64)\n    pos_rank_sum = float(ranks[y].sum())\n    return float((pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))\n\n\ndef validate_reused_species(metadata: pd.DataFrame, args: argparse.Namespace, reports_dir: Path) -> pd.DataFrame:\n    rows_out = []\n    try:\n        detector = cv2.SIFT_create(nfeatures=1100, contrastThreshold=0.012, edgeThreshold=12)\n    except Exception:\n        print(\"[warn] SIFT unavailable; reused-species validation skipped.\")\n        return pd.DataFrame()\n    train_all = metadata[metadata[\"split\"].eq(\"train\")].copy()\n    for species in REUSED_SPECIES:\n        print(f\"[{species}] 2025-style local validation\")\n        train_rows = train_all[train_all[\"species_id\"].eq(species)].copy()\n        train_rows = sample_train_rows(train_rows, args.reused_max_train_images, args.reused_max_per_identity)\n        if args.smoke:\n            train_rows = train_rows.head(44)\n        records = train_rows.to_dict(\"records\")\n        if not records:\n            continue\n        feats = {}\n        failures = 0\n        for idx, row in enumerate(records, start=1):\n            try:\n                feats[int(row[\"image_id\"])] = extract_local_features(row, species, args.max_side, detector)\n            except Exception as exc:\n                failures += 1\n                feats[int(row[\"image_id\"])] = ([], None, np.zeros((8, 8), dtype=np.uint8))\n                print(f\"[warn] {species} feature fail image_id={row.get('image_id')}: {exc}\")\n            if idx % 100 == 0:\n                print(f\"[{species}] local features {idx}/{len(records)}\")\n        pairs = sample_validation_pairs(records, min(args.reused_train_pair_budget, 350 if args.smoke else args.reused_train_pair_budget))\n        score_rows = []\n        identity_by_id = {int(r[\"image_id\"]): str(r.get(\"identity\", \"\")) for r in records}\n        for a, b in pairs:\n            s = local_pair_score(feats[a], feats[b], species)\n            same = bool(identity_by_id[a] and identity_by_id[a] == identity_by_id[b])\n            score_rows.append({\"species\": species, \"image_id_a\": a, \"image_id_b\": b, \"same_identity\": same, **s})\n        pair_df = pd.DataFrame(score_rows)\n        pair_df.to_csv(reports_dir / f\"{VERSION}_{species}_train_local_pair_scores.csv\", index=False)\n        if pair_df.empty:\n            continue\n        y = pair_df[\"same_identity\"].astype(bool).to_numpy()\n        scores = pair_df[\"score\"].astype(float).to_numpy()\n        row = {\n            \"species\": species,\n            \"n_pairs\": int(len(pair_df)),\n            \"n_positive\": int(y.sum()),\n            \"n_negative\": int((~y).sum()),\n            \"auc\": auc_rank(y, scores),\n            \"feature_failures\": int(failures),\n            \"median_inliers_positive\": float(pair_df.loc[pair_df[\"same_identity\"], \"inliers\"].median()) if y.any() else 0.0,\n            \"median_inliers_negative\": float(pair_df.loc[~pair_df[\"same_identity\"], \"inliers\"].median()) if (~y).any() else 0.0,\n        }\n        for thr in [0.35, 0.45, 0.55, 0.65]:\n            pred = scores >= thr\n            tp = int((pred & y).sum())\n            fp = int((pred & (~y)).sum())\n            fn = int(((~pred) & y).sum())\n            row[f\"precision_at_{thr:.2f}\"] = float(tp / max(1, tp + fp))\n            row[f\"recall_at_{thr:.2f}\"] = float(tp / max(1, tp + fn))\n            row[f\"accepted_at_{thr:.2f}\"] = int(pred.sum())\n        rows_out.append(row)\n    return pd.DataFrame(rows_out)\n\n\ndef build_submission(current: pd.DataFrame, test_rows: pd.DataFrame, texas_labels: dict[int, str], out_path: Path) -> pd.DataFrame:\n    sub = current.copy()\n    texas_ids = set(test_rows.loc[test_rows[\"species_id\"].eq(TEXAS), \"image_id\"].astype(int).tolist())\n    current_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(\n        lambda i: texas_labels.get(i, current_map[i]) if i in texas_ids else current_map[i]\n    )\n    sub.to_csv(out_path, index=False)\n    return sub\n\n\ndef summarize_submission(sub: pd.DataFrame, current: pd.DataFrame, test_rows: pd.DataFrame, variant: str) -> list[dict]:\n    cur_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    sub_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    rows = []\n    for species in core.SPECIES_ORDER:\n        ids = test_rows.loc[test_rows[\"species_id\"].eq(species), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": int(len(ids)),\n                \"n_clusters\": int(counts.shape[0]),\n                \"singletons\": int((counts == 1).sum()),\n                \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n                \"rows_changed_vs_current\": int(sum(1 for i in ids if sub_map[i] != cur_map[i])),\n            }\n        )\n    return rows\n\n\ndef heat_to_image(heat: np.ndarray, mask: np.ndarray) -> Image.Image:\n    h = np.clip(heat, 0, 1)\n    arr = np.zeros((h.shape[0], h.shape[1], 3), dtype=np.uint8)\n    arr[:, :, 0] = np.clip(h * 255, 0, 255).astype(np.uint8)\n    arr[:, :, 1] = np.clip((1.0 - h) * 120, 0, 255).astype(np.uint8)\n    arr[:, :, 2] = np.clip((1.0 - h) * 90, 0, 255).astype(np.uint8)\n    arr[mask == 0] = BACKGROUND\n    return Image.fromarray(arr)\n\n\ndef draw_texas_item(item: TexasDotItem, size: tuple[int, int]) -> Image.Image:\n    rgb = item.belly_rgb.copy()\n    mask = item.belly_mask\n    dim = (rgb.astype(np.float32) * 0.40 + BACKGROUND.astype(np.float32) * 0.60).astype(np.uint8)\n    rgb[mask == 0] = dim[mask == 0]\n    contours, _ = cv2.findContours(np.where(mask > 0, 255, 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    cv2.drawContours(rgb, contours, -1, (30, 255, 70), 2)\n    h, w = mask.shape[:2]\n    for p in item.dot_points[:160]:\n        x = int(round(float(p[0]) * max(1, w - 1)))\n        y = int(round(float(p[1]) * max(1, h - 1)))\n        cv2.circle(rgb, (x, y), 2, (255, 30, 30), 1, cv2.LINE_AA)\n    img = Image.fromarray(rgb)\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (18, 18, 18))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef thumb(path: str, size: tuple[int, int]) -> Image.Image:\n    try:\n        img = Image.open(path).convert(\"RGB\")\n    except Exception:\n        img = Image.new(\"RGB\", size, (25, 25, 25))\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (18, 18, 18))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef save_texas_preview(items: list[TexasDotItem], out_path: Path, limit: int) -> None:\n    chosen = items[:limit]\n    if not chosen:\n        return\n    tile_w, tile_h = 190, 190\n    label_h = 22\n    cols = 4\n    canvas = Image.new(\"RGB\", (cols * tile_w, len(chosen) * (tile_h + label_h)), (18, 18, 18))\n    draw = ImageDraw.Draw(canvas)\n    for r, item in enumerate(chosen):\n        y = r * (tile_h + label_h)\n        panels = [\n            thumb(item.source_path, (tile_w, tile_h)),\n            thumb(item.view_path, (tile_w, tile_h)),\n            draw_texas_item(item, (tile_w, tile_h)),\n            heat_to_image(item.dot_heat, item.belly_mask),\n        ]\n        labels = [f\"orig {item.image_id}\", item.view_source[:20], \"belly dots\", \"dot heat\"]\n        for c, panel in enumerate(panels):\n            panel = panel.copy()\n            panel.thumbnail((tile_w, tile_h), Image.Resampling.BILINEAR)\n            x = c * tile_w\n            draw.text((x + 5, y + 4), labels[c], fill=(245, 240, 145))\n            canvas.paste(panel, (x + (tile_w - panel.width) // 2, y + label_h + (tile_h - panel.height) // 2))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef save_pair_preview(items: list[TexasDotItem], pair_scores: pd.DataFrame, out_path: Path, limit: int, cross_only: bool = False) -> None:\n    if pair_scores.empty:\n        return\n    g = pair_scores.copy()\n    if cross_only:\n        g = g[~g[\"same_current_cluster\"].astype(bool)]\n    g = g.sort_values(\"score\", ascending=False).head(limit)\n    if g.empty:\n        return\n    by_id = {it.image_id: it for it in items}\n    panel_w, panel_h = 840, 260\n    rows = []\n    for edge in g.to_dict(\"records\"):\n        a = by_id.get(int(edge[\"image_id_a\"]))\n        b = by_id.get(int(edge[\"image_id_b\"]))\n        if a is None or b is None:\n            continue\n        pa = heat_to_image(a.dot_heat, a.belly_mask)\n        pb = heat_to_image(b.dot_heat, b.belly_mask)\n        pa.thumbnail((300, 220), Image.Resampling.BILINEAR)\n        pb.thumbnail((300, 220), Image.Resampling.BILINEAR)\n        row_img = Image.new(\"RGB\", (panel_w, panel_h), (18, 18, 18))\n        draw = ImageDraw.Draw(row_img)\n        text = (\n            f\"{a.image_id} vs {b.image_id} score={float(edge['score']):.3f} \"\n            f\"corr={float(edge['corr']):.3f} pts={float(edge['point_score']):.3f} \"\n            f\"stack={float(edge['stack_gain']):.3f} same_current={edge['same_current_cluster']}\"\n        )\n        draw.text((6, 5), text, fill=(255, 240, 120))\n        row_img.paste(pa, (12, 35))\n        row_img.paste(pb, (440, 35))\n        rows.append(row_img)\n    canvas = Image.new(\"RGB\", (panel_w, panel_h * len(rows)), (18, 18, 18))\n    for idx, row in enumerate(rows):\n        canvas.paste(row, (0, idx * panel_h))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef main() -> None:\n    random.seed(SEED)\n    np.random.seed(SEED)\n    args = parse_args()\n    if args.smoke:\n        args.texas_pair_budget = min(args.texas_pair_budget or 600, 600)\n        args.reused_train_pair_budget = min(args.reused_train_pair_budget, 260)\n        args.reused_max_train_images = min(args.reused_max_train_images, 48)\n        args.save_visualizations = True\n\n    data_root = find_data_root(args.data_root)\n    sam_manifest = find_sam_manifest(args.sam_manifest)\n    metadata, manifest_info = prepare_metadata(data_root, sam_manifest)\n    metadata = metadata[metadata[\"species_id\"].isin(core.SPECIES_ORDER)].copy()\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    sub_dir = output_root / \"submissions\"\n    viz_dir = output_root / \"visualizations\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    sub_dir.mkdir(parents=True, exist_ok=True)\n    if args.save_visualizations:\n        viz_dir.mkdir(parents=True, exist_ok=True)\n\n    current_best_path = find_current_best(args.current_best_submission, data_root)\n    current = core.load_submission(current_best_path)\n    current_labels = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"current_best={current_best_path}\")\n    print(f\"output_root={output_root}\")\n    print(json.dumps(manifest_info, indent=2))\n\n    texas_rows = test_rows[test_rows[\"species_id\"].eq(TEXAS)].sort_values(\"image_id\").copy()\n    if args.smoke:\n        texas_rows = texas_rows.head(36)\n    texas_items: list[TexasDotItem] = []\n    for idx, row in enumerate(texas_rows.to_dict(\"records\"), start=1):\n        image_id = int(row[\"image_id\"])\n        try:\n            texas_items.append(texas_belly_template(row, current_labels[image_id], args))\n        except Exception as exc:\n            print(f\"[warn] Texas template failed image_id={image_id}: {exc}\")\n        if idx % 75 == 0:\n            print(f\"[Texas astro-dot] templates {idx}/{len(texas_rows)}\")\n\n    if args.save_visualizations:\n        save_texas_preview(texas_items, viz_dir / f\"{VERSION}_Texas_template_preview.jpg\", args.visual_limit)\n\n    texas_pair_scores = score_all_texas_pairs(texas_items, args.texas_pair_budget)\n    texas_pair_scores.to_csv(reports_dir / f\"{VERSION}_Texas_pair_scores.csv\", index=False)\n    if args.save_visualizations:\n        save_pair_preview(texas_items, texas_pair_scores, viz_dir / f\"{VERSION}_Texas_top_pairs_all.jpg\", max(5, args.visual_limit // 2), False)\n        save_pair_preview(texas_items, texas_pair_scores, viz_dir / f\"{VERSION}_Texas_top_pairs_cross_cluster.jpg\", max(5, args.visual_limit // 2), True)\n\n    reused_report = pd.DataFrame()\n    if not args.skip_reused_validation:\n        reused_report = validate_reused_species(metadata, args, reports_dir)\n        reused_report.to_csv(reports_dir / f\"{VERSION}_reused_species_2025style_validation.csv\", index=False)\n\n    variants = [\"split_strict\", \"merge_ultra\", \"splitmerge_guarded\", \"splitmerge_swing\"]\n    candidate_rows: list[dict] = []\n    texas_variant_rows: list[dict] = []\n    texas_current = {it.image_id: it.current_cluster for it in texas_items}\n    for variant in variants:\n        texas_labels = texas_variant_labels(texas_items, texas_pair_scores, variant)\n        texas_variant_rows.append(summarize_texas_variant(texas_labels, texas_pair_scores, variant, texas_current))\n        out_path = sub_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub = build_submission(current, test_rows, texas_labels, out_path)\n        candidate_rows.extend(summarize_submission(sub, current, test_rows, variant))\n        print(f\"wrote {out_path}\")\n\n    texas_variant_report = pd.DataFrame(texas_variant_rows)\n    candidate_report = pd.DataFrame(candidate_rows)\n    texas_variant_report.to_csv(reports_dir / f\"{VERSION}_texas_variant_report.csv\", index=False)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"current_best\": str(current_best_path),\n        \"manifest_info\": manifest_info,\n        \"outputs\": {\n            \"reports_dir\": str(reports_dir),\n            \"submissions_dir\": str(sub_dir),\n            \"visualizations_dir\": str(viz_dir),\n        },\n        \"texas_items\": int(len(texas_items)),\n        \"texas_pairs\": int(len(texas_pair_scores)),\n        \"texas_variant_report\": texas_variant_report.to_dict(\"records\"),\n        \"candidate_report\": candidate_report.to_dict(\"records\"),\n        \"reused_validation\": reused_report.to_dict(\"records\") if not reused_report.empty else [],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    print(\"\\nTexas variant report:\")\n    print(texas_variant_report.to_string(index=False))\n    print(\"\\nCandidate report:\")\n    print(candidate_report.to_string(index=False))\n    if not reused_report.empty:\n        print(\"\\nReused-species 2025-style validation:\")\n        print(reused_report.to_string(index=False))\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("texas_astrodot_nooval_v20260427.py").write_text("from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nimport texas_astrodot_2025reuse_v20260426 as base\n\n\nVERSION = \"texas_astrodot_nooval_v20260427\"\n\n\ndef align_vertical_no_flip(rgb: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:\n    \"\"\"Canonicalize Texas crops vertically without head/tail flipping.\n\n    Field photos for this species already have the head at the top. The base\n    helper flipped images when lower mask bands looked wider, but that can turn\n    correctly oriented belly-dot fingerprints upside down.\n    \"\"\"\n    crop_rgb, crop_mask = base.crop_to_mask(rgb, mask, 0.08)\n    angle = base.core.pca_angle_degrees(crop_mask)\n    rotate_angle = 90.0 - angle\n    if abs(rotate_angle) > 1.5:\n        crop_rgb, crop_mask = base.core.rotate_bound(crop_rgb, crop_mask, rotate_angle)\n        crop_rgb, crop_mask = base.crop_to_mask(crop_rgb, crop_mask, 0.04)\n    h, _ = crop_mask.shape[:2]\n    widths = []\n    for yf in [0.18, 0.30, 0.70, 0.84]:\n        y = int(np.clip(round(h * yf), 0, h - 1))\n        xs = np.where(crop_mask[y] > 0)[0]\n        widths.append(float(xs.max() - xs.min() + 1) if len(xs) else 0.0)\n    top_width = max(widths[:2])\n    bottom_width = max(widths[2:])\n    return crop_rgb, crop_mask, {\n        \"pca_angle\": float(angle),\n        \"rotate_angle\": float(rotate_angle),\n        \"top_width\": float(top_width),\n        \"bottom_width\": float(bottom_width),\n        \"flipped_vertical\": False,\n        \"orientation_rule\": \"head_already_top_no_flip\",\n    }\n\n\ndef texas_belly_template_no_oval(row: dict, current_cluster: str, args: argparse.Namespace) -> base.TexasDotItem:\n    \"\"\"Texas v2: use the full aligned SAM-clean crop, not an oval belly cut.\n\n    The Texas photos are already mostly standardized ventral views. The older\n    ellipse mask removed some real belly/peripheral dot evidence, especially on\n    wide bellies and side-edge dot patterns. This version keeps the complete\n    cleaned foreground mask after alignment and canvas resize.\n    \"\"\"\n    rgb, mask, quality = base.read_rgb_mask(row, args.max_side)\n    aligned_rgb, aligned_mask, debug = align_vertical_no_flip(rgb, mask)\n    w = int(args.texas_canvas_w)\n    h = int(args.texas_canvas_h)\n    belly_rgb = cv2.resize(aligned_rgb, (w, h), interpolation=cv2.INTER_AREA)\n    belly_mask = cv2.resize(aligned_mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    belly_mask = base.clean_mask(belly_mask)\n\n    coverage = float(belly_mask.mean() / 255.0)\n    if coverage < 0.035:\n        # If the mask is unexpectedly missing, use the full crop instead of an\n        # artificial oval. This preserves the user's visual observation that the\n        # SAM-clean crop itself is the useful standardized object.\n        belly_mask = np.full((h, w), 255, dtype=np.uint8)\n        quality *= 0.68\n        coverage = 1.0\n    elif coverage > 0.92:\n        # Full masks are okay here; just reduce quality slightly because they\n        # may contain a little background residue.\n        quality *= 0.94\n\n    debug = dict(debug)\n    debug[\"mask_mode\"] = \"full_aligned_crop_no_oval\"\n    debug[\"mask_coverage\"] = coverage\n\n    heat = base.texas_dot_heat(belly_rgb, belly_mask)\n    points = base.dot_points_from_heat(heat, belly_mask)\n    small = cv2.resize(heat, (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    mask_small = cv2.resize((belly_mask > 0).astype(np.float32), (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    vector = base.normalize(np.concatenate([small * mask_small, mask_small * 0.18]).astype(np.float32))\n    quality *= min(1.0, 0.72 + 0.28 * min(1.0, len(points) / 45.0))\n\n    return base.TexasDotItem(\n        image_id=int(row[\"image_id\"]),\n        current_cluster=str(current_cluster),\n        source_path=str(row.get(\"source_path\", \"\")),\n        view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n        view_source=str(row.get(\"view_source\", \"\")),\n        belly_rgb=belly_rgb,\n        belly_mask=belly_mask,\n        dot_heat=heat,\n        dot_points=points,\n        vector=vector,\n        quality=float(quality),\n        debug=debug,\n    )\n\n\ndef texas_pair_score_no_flip(a: base.TexasDotItem, b: base.TexasDotItem) -> dict:\n    \"\"\"Score Texas belly dots without vertical flip search.\"\"\"\n    desc_cos = float(np.dot(a.vector, b.vector))\n    h, w = a.dot_heat.shape[:2]\n    best = {\n        \"score\": 0.0,\n        \"corr\": 0.0,\n        \"overlap\": 0.0,\n        \"stack_gain\": 0.0,\n        \"point_score\": 0.0,\n        \"descriptor_cosine\": desc_cos,\n        \"dx\": 0,\n        \"dy\": 0,\n        \"transform\": \"identity\",\n    }\n    common_mask = np.where((a.belly_mask > 0), 1.0, 0.0).astype(np.float32)\n    base_dx, base_dy = base.phase_shift(a.dot_heat, b.dot_heat, common_mask)\n    candidates = {(0, 0), (base_dx, base_dy)}\n    for dx0, dy0 in [(base_dx, base_dy), (0, 0)]:\n        for ddx in (-8, 0, 8):\n            for ddy in (-8, 0, 8):\n                candidates.add((int(np.clip(dx0 + ddx, -24, 24)), int(np.clip(dy0 + ddy, -24, 24))))\n    for dx, dy in candidates:\n        corr, overlap, stack_gain = base.masked_corr_and_stack(a.dot_heat, a.belly_mask, b.dot_heat, b.belly_mask, dx, dy)\n        if overlap < 0.045:\n            continue\n        point_score = base.chamfer_dot_score(a.dot_points, b.dot_points, dx / max(1, w - 1), dy / max(1, h - 1))\n        fused = (\n            0.34 * corr\n            + 0.31 * point_score\n            + 0.23 * stack_gain\n            + 0.12 * max(0.0, desc_cos)\n        )\n        fused *= min(a.quality, b.quality)\n        if fused > best[\"score\"]:\n            best = {\n                \"score\": float(fused),\n                \"corr\": float(corr),\n                \"overlap\": float(overlap),\n                \"stack_gain\": float(stack_gain),\n                \"point_score\": float(point_score),\n                \"descriptor_cosine\": desc_cos,\n                \"dx\": int(dx),\n                \"dy\": int(dy),\n                \"transform\": \"identity\",\n            }\n    best[\"points_a\"] = int(len(a.dot_points))\n    best[\"points_b\"] = int(len(b.dot_points))\n    best[\"quality_a\"] = float(a.quality)\n    best[\"quality_b\"] = float(b.quality)\n    best[\"same_current_cluster\"] = bool(a.current_cluster == b.current_cluster)\n    return best\n\n\ndef main() -> None:\n    base.VERSION = VERSION\n    base.align_vertical = align_vertical_no_flip\n    base.texas_belly_template = texas_belly_template_no_oval\n    base.texas_pair_score = texas_pair_score_no_flip\n    base.main()\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("sam_yolo_species_submission_v20260427.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nVERSION = \"sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427\"\nSEED = 20260427\nIMG_SIZE = 512\nSPECIES_ORDER = [\n    \"LynxID2025\",\n    \"SalamanderID2025\",\n    \"SeaTurtleID2022\",\n    \"TexasHornedLizards\",\n]\nREID_SPECIES = [\"LynxID2025\", \"SalamanderID2025\", \"SeaTurtleID2022\"]\nTEXAS = \"TexasHornedLizards\"\nNEUTRAL_GREY_RGB = np.array([128, 128, 128], dtype=np.uint8)\nBLACK_RGB = np.array([0, 0, 0], dtype=np.uint8)\nBACKGROUND_RGB = NEUTRAL_GREY_RGB\n\n\n@dataclass\nclass FeatureItem:\n    image_id: int\n    species: str\n    identity: str\n    cluster: str\n    view_path: str\n    source_path: str\n    orientation: str\n    body_kind: str\n    low_light: bool\n    keypoints: list\n    desc: np.ndarray | None\n    vec: np.ndarray\n    quality: float\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> bool:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return False\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return True\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Consume the SAM+YOLO fused view manifest and create species-specific \"\n            \"AnimalCLEF submission candidates from independent train-calibrated \"\n            \"pattern graphs. Existing submissions are optional diagnostics only.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--view-manifest\", type=str, default=None)\n    parser.add_argument(\"--reference-submission\", type=str, default=None, help=\"Optional diagnostics only; not auto-discovered.\")\n    parser.add_argument(\"--base-submission\", type=str, default=None, help=\"Deprecated diagnostics alias for --reference-submission.\")\n    parser.add_argument(\"--output-root\", type=str, default=\"/kaggle/working/animalclef_sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427\")\n    parser.add_argument(\"--img-size\", type=int, default=IMG_SIZE, help=\"Global square canvas size after aspect-ratio padding.\")\n    parser.add_argument(\"--max-side\", type=int, default=760)\n    parser.add_argument(\"--top-k\", type=int, default=48)\n    parser.add_argument(\"--pair-budget-per-species\", type=int, default=85000)\n    parser.add_argument(\"--train-images-per-species\", type=int, default=850)\n    parser.add_argument(\"--train-pairs-per-species\", type=int, default=3200)\n    parser.add_argument(\"--texas-pair-budget\", type=int, default=18000)\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--visual-limit\", type=int, default=20)\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n            Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\animal-clef-2026\"),\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_view_manifest(user_value: str | None) -> Path:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam_yolo_views_v20260427/reports/view_manifest_sam_yolo_fused_v20260427.csv\"),\n        Path(\"/kaggle/input/animalclef2026-sam-yolo-view-factory-v20260427/animalclef_sam_yolo_views_v20260427/reports/view_manifest_sam_yolo_fused_v20260427.csv\"),\n        Path(\"/kaggle/input/animalclef2026-sam-yolo-view-factory-v20260427/reports/view_manifest_sam_yolo_fused_v20260427.csv\"),\n        Path.cwd() / \"animalclef_sam_yolo_views_v20260427\" / \"reports\" / \"view_manifest_sam_yolo_fused_v20260427.csv\",\n        Path.cwd().parent / \"animalclef_sam_yolo_views_v20260427\" / \"reports\" / \"view_manifest_sam_yolo_fused_v20260427.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent, Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\")]:\n        if not base.exists():\n            continue\n        try:\n            matches = list(base.rglob(\"view_manifest_sam_yolo_fused_v20260427.csv\"))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda x: len(str(x)))\n            return matches[0].resolve()\n    raise FileNotFoundError(\"Could not locate view_manifest_sam_yolo_fused_v20260427.csv.\")\n\n\ndef find_reference_submission(user_value: str | None, data_root: Path) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    return None\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns.\")\n    df = df[[\"image_id\", \"cluster\"]].copy()\n    df[\"image_id\"] = df[\"image_id\"].astype(int)\n    df[\"cluster\"] = df[\"cluster\"].astype(str)\n    return df\n\n\ndef validate_submission(sub: pd.DataFrame, sample: pd.DataFrame) -> None:\n    a = set(sub[\"image_id\"].astype(int))\n    b = set(sample[\"image_id\"].astype(int))\n    if a != b or len(sub) != len(sample):\n        raise ValueError(f\"Bad submission: len={len(sub)} expected={len(sample)} missing={sorted(b-a)[:5]} extra={sorted(a-b)[:5]}\")\n    if sub[\"cluster\"].isna().any():\n        raise ValueError(\"Submission contains null clusters.\")\n\n\ndef remap_path(path_value: object, manifest_root: Path) -> str:\n    if path_value is None or pd.isna(path_value):\n        return \"\"\n    s = str(path_value).strip()\n    if not s:\n        return \"\"\n    p = Path(s)\n    if p.exists():\n        return str(p.resolve())\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\"animalclef_sam_yolo_views_v20260427/\", \"views/\", \"yolo_pred_mask/\"]\n    base_root = manifest_root.parent.parent if manifest_root.parent.name == \"reports\" else manifest_root.parent\n    for marker in markers:\n        if marker not in normalized:\n            continue\n        rel = normalized.split(marker, 1)[1]\n        if marker != \"animalclef_sam_yolo_views_v20260427/\":\n            rel = marker + rel\n        candidate = base_root / Path(rel)\n        if candidate.exists():\n            return str(candidate.resolve())\n    return s\n\n\ndef prepare_view_manifest(view_manifest: Path, data_root: Path) -> pd.DataFrame:\n    df = pd.read_csv(view_manifest)\n    metadata = pd.read_csv(data_root / \"metadata.csv\")\n    if \"species_id\" not in metadata.columns:\n        metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n    metadata = metadata[metadata[\"species_id\"].isin(SPECIES_ORDER)].copy()\n    keep_meta = [c for c in [\"image_id\", \"identity\", \"orientation\", \"path\", \"split\", \"dataset\", \"species_id\"] if c in metadata.columns]\n    if \"identity\" in df.columns:\n        df = df.drop(columns=[\"identity\"], errors=\"ignore\")\n    if \"orientation\" in df.columns:\n        df = df.drop(columns=[\"orientation\"], errors=\"ignore\")\n    if \"split\" in df.columns:\n        df = df.drop(columns=[\"split\"], errors=\"ignore\")\n    df = df.merge(metadata[keep_meta], on=\"image_id\", how=\"left\", suffixes=(\"\", \"_meta\"))\n    if \"dataset\" in df.columns:\n        df[\"species_id\"] = df[\"species_id\"].fillna(df[\"dataset\"])\n    df[\"species_id\"] = df[\"species_id\"].astype(str)\n    inferred_split = pd.Series(\n        np.where(df[\"path\"].astype(str).str.contains(\"/test/\"), \"test\", \"train\"),\n        index=df.index,\n    )\n    df[\"split\"] = df[\"split\"].fillna(inferred_split).astype(str)\n    df[\"identity\"] = df[\"identity\"].fillna(\"\").astype(str)\n    df[\"orientation\"] = df[\"orientation\"].fillna(\"unknown\").astype(str)\n    if \"original_path\" not in df.columns or df[\"original_path\"].isna().any():\n        df[\"original_path\"] = df[\"path\"].map(lambda p: str(data_root / str(p)))\n    for col in [\n        \"original_path\",\n        \"view_original_path\",\n        \"view_sam_clean_path\",\n        \"view_yolo_crop_path\",\n        \"view_sam_yolo_union_path\",\n        \"view_sam_yolo_intersection_path\",\n        \"view_species_final_path\",\n        \"fused_mask_path\",\n    ]:\n        if col not in df.columns:\n            df[col] = \"\"\n        df[col] = df[col].map(lambda x: remap_path(x, view_manifest))\n    # Reused Kaggle outputs often preserve /kaggle/input/... original image\n    # paths. When this notebook is run locally, or when Kaggle mounts the\n    # competition dataset under a different alias, rebuild original_path from\n    # metadata instead of letting Texas full-canvas masking fail.\n    if \"path\" in df.columns:\n        rebuilt_original = df[\"path\"].map(lambda p: str(data_root / str(p)))\n        missing_original = df[\"original_path\"].fillna(\"\").astype(str).map(lambda p: (not p) or (not Path(p).exists()))\n        df.loc[missing_original, \"original_path\"] = rebuilt_original[missing_original]\n    present = set(df[\"image_id\"].astype(int))\n    missing = metadata[~metadata[\"image_id\"].astype(int).isin(present)].copy()\n    if not missing.empty:\n        # Smoke/local partial manifests should still produce a valid submission.\n        # For missing rows, fall back to original images and singleton clustering.\n        extra = missing[keep_meta].copy()\n        extra[\"original_path\"] = extra[\"path\"].map(lambda p: str(data_root / str(p)))\n        for col in [\n            \"view_original_path\",\n            \"view_sam_clean_path\",\n            \"view_yolo_crop_path\",\n            \"view_sam_yolo_union_path\",\n            \"view_sam_yolo_intersection_path\",\n            \"view_species_final_path\",\n        ]:\n            extra[col] = extra[\"original_path\"]\n        extra[\"fused_mask_path\"] = \"\"\n        extra[\"fusion_decision\"] = \"original_missing_fused_manifest\"\n        extra[\"final_view_kind\"] = \"original_fallback_missing_manifest\"\n        extra[\"sam_ok\"] = False\n        extra[\"yolo_ok\"] = False\n        extra[\"sam_yolo_iou\"] = 0.0\n        df = pd.concat([df, extra], ignore_index=True, sort=False)\n    return df[df[\"species_id\"].isin(SPECIES_ORDER)].copy()\n\n\ndef image_path_for_species(row: dict, species: str) -> str:\n    if species == \"LynxID2025\":\n        # Lynx camera-trap frames are often already subject-dominant and dark.\n        # Use the original frame as the signal source; the fused mask is only\n        # a subject guide for preprocessing.\n        return str(row.get(\"original_path\") or row.get(\"view_original_path\") or row.get(\"view_species_final_path\"))\n    if species == \"SalamanderID2025\":\n        # Salamander identity lives in the black/yellow pattern and the user\n        # observed orientation is informative. Use SAM-clean output, but do not\n        # rotate or flip it later.\n        return str(row.get(\"view_sam_clean_path\") or row.get(\"view_species_final_path\") or row.get(\"view_sam_yolo_union_path\"))\n    if species == \"SeaTurtleID2022\":\n        return str(row.get(\"view_sam_yolo_intersection_path\") or row.get(\"view_species_final_path\") or row.get(\"view_sam_clean_path\"))\n    # Texas uses the final SAM+YOLO view; the astro-dot template below enforces\n    # no oval and no vertical flip.\n    return str(row.get(\"view_species_final_path\") or row.get(\"view_sam_yolo_union_path\") or row.get(\"view_sam_clean_path\") or row.get(\"original_path\"))\n\n\ndef read_rgb(path: str, max_side: int) -> np.ndarray:\n    img = Image.open(path).convert(\"RGB\")\n    w, h = img.size\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        img = img.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))), Image.Resampling.BILINEAR)\n    return np.asarray(img)\n\n\ndef read_mask(path: str, shape: tuple[int, int]) -> np.ndarray:\n    if path and Path(path).exists():\n        mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)\n        if mask is not None:\n            if mask.shape[:2] != shape:\n                mask = cv2.resize(mask, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)\n            return np.where(mask > 16, 255, 0).astype(np.uint8)\n    return np.full(shape, 255, dtype=np.uint8)\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad_frac: float) -> tuple[np.ndarray, np.ndarray]:\n    ys, xs = np.where(mask > 0)\n    if len(xs) < 20:\n        return rgb, mask\n    h, w = mask.shape[:2]\n    x0, x1 = int(xs.min()), int(xs.max())\n    y0, y1 = int(ys.min()), int(ys.max())\n    pad = int(round(max(x1 - x0 + 1, y1 - y0 + 1) * pad_frac))\n    x0 = max(0, x0 - pad)\n    y0 = max(0, y0 - pad)\n    x1 = min(w - 1, x1 + pad)\n    y1 = min(h - 1, y1 + pad)\n    return rgb[y0 : y1 + 1, x0 : x1 + 1].copy(), mask[y0 : y1 + 1, x0 : x1 + 1].copy()\n\n\ndef square_pad_resize(\n    rgb: np.ndarray,\n    mask: np.ndarray,\n    img_size: int,\n    pad_rgb: np.ndarray = BLACK_RGB,\n) -> tuple[np.ndarray, np.ndarray]:\n    h, w = rgb.shape[:2]\n    side = max(h, w, 1)\n    canvas = np.zeros((side, side, 3), dtype=np.uint8)\n    canvas[:] = pad_rgb.reshape(1, 1, 3)\n    mask_canvas = np.zeros((side, side), dtype=np.uint8)\n    y0 = (side - h) // 2\n    x0 = (side - w) // 2\n    canvas[y0 : y0 + h, x0 : x0 + w] = rgb\n    mask_canvas[y0 : y0 + h, x0 : x0 + w] = mask\n    out = cv2.resize(canvas, (img_size, img_size), interpolation=cv2.INTER_AREA)\n    out_mask = cv2.resize(mask_canvas, (img_size, img_size), interpolation=cv2.INTER_NEAREST)\n    return out, out_mask\n\n\ndef canonical_orientation(value: object) -> str:\n    s = str(value or \"unknown\").strip().lower()\n    if not s or s in {\"nan\", \"na\", \"none\"}:\n        return \"unknown\"\n    return s\n\n\ndef rotate_by_orientation(rgb: np.ndarray, species: str, orientation: str) -> np.ndarray:\n    \"\"\"Use metadata orientation where it is semantically reliable.\n\n    Salamander images are often randomly rotated. The 2025 solutions explicitly\n    normalized salamander orientation; doing it here lets head-only and full-body\n    crops compare in a common frame. Lynx/SeaTurtle orientation is used as a\n    matching rule instead of physically rotating the image.\n    \"\"\"\n    # Final correction: do not physically rotate or flip any species here.\n    # Salamander original orientation is treated as a pattern clue, not noise.\n    return rgb\n\n\ndef enhance_lynx_low_light(rgb: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, bool, float]:\n    \"\"\"Adaptive low-light enhancement for dark Lynx camera-trap frames.\"\"\"\n    if rgb.size == 0:\n        return rgb, False, 0.0\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n    l = lab[:, :, 0]\n    median_l = float(np.median(l))\n    p90_l = float(np.percentile(l, 90))\n    low_light = median_l < 72.0 or p90_l < 118.0\n    if not low_light:\n        # Still apply a mild CLAHE to make dark spots less dependent on exposure.\n        enhanced = cv2.createCLAHE(clipLimit=1.6, tileGridSize=(8, 8)).apply(l)\n        lab[:, :, 0] = np.where(mask > 0, enhanced, l).astype(np.uint8)\n        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), False, median_l\n\n    l_eq = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(l)\n    l_eq = np.where(mask > 0, l_eq, l).astype(np.uint8)\n    rgb_eq = cv2.cvtColor(np.dstack([l_eq, lab[:, :, 1], lab[:, :, 2]]).astype(np.uint8), cv2.COLOR_LAB2RGB)\n    # Gamma < 1 brightens the midtones without completely flattening spots.\n    gamma = float(np.clip(0.58 + median_l / 180.0, 0.55, 0.92))\n    lut = np.array([np.clip(((i / 255.0) ** gamma) * 255.0, 0, 255) for i in range(256)], dtype=np.uint8)\n    rgb_gamma = cv2.LUT(rgb_eq, lut)\n    rgb_gamma = np.where(mask[:, :, None] > 0, rgb_gamma, rgb).astype(np.uint8)\n    return rgb_gamma, True, median_l\n\n\ndef preprocess_rgb_for_species(\n    rgb: np.ndarray,\n    mask: np.ndarray,\n    species: str,\n    orientation: str,\n    img_size: int,\n) -> tuple[np.ndarray, np.ndarray, bool, float]:\n    rgb = rotate_by_orientation(rgb, species, orientation)\n    if species == \"LynxID2025\":\n        rgb, low_light, light_level = enhance_lynx_low_light(rgb, mask)\n        rgb, mask = crop_to_mask(rgb, mask, 0.10)\n        rgb, mask = square_pad_resize(rgb, mask, img_size, BLACK_RGB)\n        return rgb, mask, bool(low_light), float(light_level)\n    if species == \"SalamanderID2025\":\n        rgb = np.where(mask[:, :, None] > 0, rgb, NEUTRAL_GREY_RGB.reshape(1, 1, 3)).astype(np.uint8)\n        rgb, mask = crop_to_mask(rgb, mask, 0.04)\n        rgb, mask = square_pad_resize(rgb, mask, img_size, NEUTRAL_GREY_RGB)\n        return rgb, mask, False, 0.0\n    if species == TEXAS:\n        rgb = np.where(mask[:, :, None] > 0, rgb, NEUTRAL_GREY_RGB.reshape(1, 1, 3)).astype(np.uint8)\n        rgb, mask = crop_to_mask(rgb, mask, 0.04)\n        rgb, mask = square_pad_resize(rgb, mask, img_size, NEUTRAL_GREY_RGB)\n        return rgb, mask, False, 0.0\n    # Sea Turtle keeps the existing clean-view route, with only the global\n    # square-pad/resize normalization applied.\n    rgb, mask = crop_to_mask(rgb, mask, 0.06)\n    rgb, mask = square_pad_resize(rgb, mask, img_size, NEUTRAL_GREY_RGB)\n    return rgb, mask, False, 0.0\n\n\ndef estimate_body_kind(rgb: np.ndarray, species: str) -> str:\n    if species != \"SalamanderID2025\" or rgb.size == 0:\n        return \"body\"\n    diff = np.abs(rgb.astype(np.int16) - BACKGROUND_RGB[None, None, :].astype(np.int16)).sum(axis=2)\n    mask = (diff > 28).astype(np.uint8) * 255\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))\n    ys, xs = np.where(mask > 0)\n    if len(xs) < 32:\n        return \"partial_or_head\"\n    w = float(xs.max() - xs.min() + 1)\n    h = float(ys.max() - ys.min() + 1)\n    elongation = max(w, h) / max(1.0, min(w, h))\n    coverage = float(mask.mean() / 255.0)\n    if elongation < 1.55 or coverage < 0.10:\n        return \"partial_or_head\"\n    return \"full_body\"\n\n\ndef orientation_compatible(a: \"FeatureItem\", b: \"FeatureItem\") -> bool:\n    if a.species != b.species:\n        return False\n    oa = canonical_orientation(a.orientation)\n    ob = canonical_orientation(b.orientation)\n    if \"unknown\" in {oa, ob}:\n        return True\n    if a.species == \"LynxID2025\":\n        sides = {\"left\", \"right\"}\n        if oa in sides and ob in sides and oa != ob:\n            return False\n        if {oa, ob} == {\"front\", \"back\"}:\n            return False\n    if a.species == \"SeaTurtleID2022\":\n        lefts = {\"left\", \"topleft\"}\n        rights = {\"right\", \"topright\"}\n        if (oa in lefts and ob in rights) or (oa in rights and ob in lefts):\n            return False\n    return True\n\n\ndef pair_context_factor(a: \"FeatureItem\", b: \"FeatureItem\") -> tuple[float, str]:\n    oa = canonical_orientation(a.orientation)\n    ob = canonical_orientation(b.orientation)\n    if not orientation_compatible(a, b):\n        return 0.0, \"orientation_block\"\n    factor = 1.0\n    relation = \"orientation_unknown_or_mixed\"\n    if oa == ob and oa != \"unknown\":\n        relation = \"same_orientation\"\n        if a.species in {\"LynxID2025\", \"SeaTurtleID2022\"}:\n            factor *= 1.05\n    elif a.species == \"LynxID2025\":\n        relation = \"weak_orientation_match\"\n        factor *= 0.92\n    if a.species == \"SalamanderID2025\":\n        if a.body_kind == b.body_kind:\n            relation += \"_same_body_kind\"\n            factor *= 1.04\n        elif \"partial_or_head\" in {a.body_kind, b.body_kind}:\n            relation += \"_head_full_mixed\"\n            factor *= 0.88\n    return float(factor), relation\n\n\ndef pattern_gray(rgb: np.ndarray, species: str) -> np.ndarray:\n    if rgb.size == 0:\n        return np.zeros((64, 64), dtype=np.uint8)\n    if species == \"SalamanderID2025\":\n        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n        yellow = lab[:, :, 2].astype(np.float32)\n        sat = hsv[:, :, 1].astype(np.float32)\n        val = hsv[:, :, 2].astype(np.float32)\n        gray = 0.46 * yellow + 0.34 * sat + 0.20 * (255.0 - val)\n    elif species == \"LynxID2025\":\n        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n        l = lab[:, :, 0].astype(np.float32)\n        inv = 255.0 - l\n        blackhat = cv2.morphologyEx(l.astype(np.uint8), cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))).astype(np.float32)\n        gray = 0.72 * inv + 0.28 * blackhat\n    elif species == \"SeaTurtleID2022\":\n        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n        gray = 0.65 * lab[:, :, 0].astype(np.float32) + 0.35 * lab[:, :, 1].astype(np.float32)\n    else:\n        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n        l = lab[:, :, 0].astype(np.float32)\n        gray = 255.0 - l\n    gray = np.clip(gray, 0, 255).astype(np.uint8)\n    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)\n\n\ndef normalize(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32)\n    n = float(np.linalg.norm(vec))\n    if n < 1e-8:\n        return vec\n    return vec / n\n\n\ndef compute_vec(rgb: np.ndarray, gray: np.ndarray) -> np.ndarray:\n    small = cv2.resize(gray, (48, 48), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0\n    small = small.reshape(-1)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    h_hist = cv2.calcHist([hsv], [0], None, [24], [0, 180]).reshape(-1)\n    s_hist = cv2.calcHist([hsv], [1], None, [16], [0, 256]).reshape(-1)\n    v_hist = cv2.calcHist([hsv], [2], None, [16], [0, 256]).reshape(-1)\n    hist = np.concatenate([h_hist, s_hist, v_hist]).astype(np.float32)\n    hist /= max(1.0, float(hist.sum()))\n    return normalize(np.concatenate([small, hist * 4.0]).astype(np.float32))\n\n\ndef make_detector():\n    try:\n        return \"sift\", cv2.SIFT_create(nfeatures=900, contrastThreshold=0.018, edgeThreshold=12)\n    except Exception:\n        return \"orb\", cv2.ORB_create(nfeatures=1200, fastThreshold=7)\n\n\ndef root_sift(desc: np.ndarray | None, detector_name: str) -> np.ndarray | None:\n    if desc is None or len(desc) == 0:\n        return None\n    if detector_name != \"sift\":\n        return desc.astype(np.uint8)\n    desc = desc.astype(np.float32)\n    desc /= np.maximum(1e-7, desc.sum(axis=1, keepdims=True))\n    return np.sqrt(desc)\n\n\ndef feature_item(row: dict, species: str, labels: dict[int, str], detector_name: str, detector, max_side: int, img_size: int) -> FeatureItem:\n    image_id = int(row[\"image_id\"])\n    orientation = canonical_orientation(row.get(\"orientation\", \"unknown\"))\n    path = image_path_for_species(row, species)\n    if not path or not Path(path).exists():\n        path = str(row.get(\"original_path\", \"\"))\n    try:\n        rgb = read_rgb(path, max_side=max_side)\n    except Exception:\n        rgb = np.zeros((96, 96, 3), dtype=np.uint8)\n    mask_path = str(row.get(\"fused_mask_path\", \"\") or row.get(\"sam_mask_path\", \"\"))\n    mask = read_mask(mask_path, rgb.shape[:2])\n    rgb, mask, low_light, light_level = preprocess_rgb_for_species(rgb, mask, species, orientation, img_size)\n    body_kind = estimate_body_kind(rgb, species)\n    gray = pattern_gray(rgb, species)\n    kps, desc = detector.detectAndCompute(gray, None)\n    if kps is None:\n        kps = []\n    desc = root_sift(desc, detector_name)\n    vec = compute_vec(rgb, gray)\n    quality = min(1.0, max(0.20, len(kps) / 320.0))\n    if species == \"LynxID2025\" and low_light:\n        quality *= 0.96\n    if species == \"SalamanderID2025\" and body_kind == \"partial_or_head\":\n        quality *= 0.92\n    return FeatureItem(\n        image_id=image_id,\n        species=species,\n        identity=str(row.get(\"identity\", \"\")),\n        cluster=str(labels.get(image_id, \"\")),\n        view_path=path,\n        source_path=str(row.get(\"original_path\", path)),\n        orientation=orientation,\n        body_kind=body_kind,\n        low_light=bool(low_light),\n        keypoints=kps,\n        desc=desc,\n        vec=vec,\n        quality=float(quality),\n    )\n\n\ndef cosine_pairs(items: list[FeatureItem], top_k: int, budget: int) -> list[tuple[int, int, float]]:\n    if len(items) < 2:\n        return []\n    ids = [it.image_id for it in items]\n    mat = np.stack([it.vec for it in items]).astype(np.float32)\n    sim = mat @ mat.T\n    np.fill_diagonal(sim, -np.inf)\n    k = min(max(1, top_k), len(items) - 1)\n    pairs: dict[tuple[int, int], float] = {}\n    for i, image_id in enumerate(ids):\n        idx = np.argpartition(-sim[i], kth=k - 1)[:k]\n        for j in idx:\n            if not orientation_compatible(items[i], items[int(j)]):\n                continue\n            a, b = sorted((int(image_id), int(ids[int(j)])))\n            pairs[(a, b)] = max(float(sim[i, int(j)]), pairs.get((a, b), -999.0))\n    out = [(a, b, s) for (a, b), s in pairs.items()]\n    out.sort(key=lambda x: x[2], reverse=True)\n    if budget and len(out) > budget:\n        out = out[:budget]\n    return out\n\n\ndef score_pair(a: FeatureItem, b: FeatureItem, detector_name: str) -> dict:\n    vec_sim = float(np.dot(a.vec, b.vec))\n    context_factor, context_relation = pair_context_factor(a, b)\n    if context_factor <= 0.0:\n        return {\n            \"vec_sim\": vec_sim,\n            \"local_score\": 0.0,\n            \"fused_score\": 0.0,\n            \"good_matches\": 0,\n            \"inliers\": 0,\n            \"inlier_ratio\": 0.0,\n            \"context_factor\": 0.0,\n            \"context_relation\": context_relation,\n            \"orientation_a\": a.orientation,\n            \"orientation_b\": b.orientation,\n            \"body_kind_a\": a.body_kind,\n            \"body_kind_b\": b.body_kind,\n            \"low_light_a\": bool(a.low_light),\n            \"low_light_b\": bool(b.low_light),\n        }\n    if a.desc is None or b.desc is None or len(a.desc) < 4 or len(b.desc) < 4:\n        return {\n            \"vec_sim\": vec_sim,\n            \"local_score\": 0.0,\n            \"fused_score\": float(context_factor * 0.40 * max(0.0, vec_sim)),\n            \"good_matches\": 0,\n            \"inliers\": 0,\n            \"inlier_ratio\": 0.0,\n            \"context_factor\": context_factor,\n            \"context_relation\": context_relation,\n            \"orientation_a\": a.orientation,\n            \"orientation_b\": b.orientation,\n            \"body_kind_a\": a.body_kind,\n            \"body_kind_b\": b.body_kind,\n            \"low_light_a\": bool(a.low_light),\n            \"low_light_b\": bool(b.low_light),\n        }\n    norm = cv2.NORM_L2 if detector_name == \"sift\" else cv2.NORM_HAMMING\n    matcher = cv2.BFMatcher(norm, crossCheck=False)\n    try:\n        knn = matcher.knnMatch(a.desc, b.desc, k=2)\n    except Exception:\n        knn = []\n    ratio = 0.80 if a.species == \"SalamanderID2025\" else 0.76\n    good = []\n    for pair in knn:\n        if len(pair) != 2:\n            continue\n        m, n = pair\n        if m.distance < ratio * max(1e-8, n.distance):\n            good.append(m)\n    inliers = 0\n    inlier_ratio = 0.0\n    if len(good) >= 4:\n        pts_a = np.float32([a.keypoints[m.queryIdx].pt for m in good]).reshape(-1, 2)\n        pts_b = np.float32([b.keypoints[m.trainIdx].pt for m in good]).reshape(-1, 2)\n        try:\n            _, mask = cv2.estimateAffinePartial2D(pts_a, pts_b, method=cv2.RANSAC, ransacReprojThreshold=5.0, maxIters=1200)\n            if mask is not None:\n                inliers = int(mask.ravel().sum())\n        except Exception:\n            inliers = 0\n    if good:\n        inlier_ratio = float(inliers / max(1, len(good)))\n    local_score = (\n        0.46 * min(1.0, inliers / 18.0)\n        + 0.28 * min(1.0, len(good) / 70.0)\n        + 0.26 * inlier_ratio\n    )\n    local_score *= min(a.quality, b.quality)\n    if a.species == \"LynxID2025\":\n        fused = 0.56 * local_score + 0.44 * max(0.0, vec_sim)\n    elif a.species == \"SalamanderID2025\":\n        if \"partial_or_head\" in {a.body_kind, b.body_kind}:\n            fused = 0.47 * local_score + 0.53 * max(0.0, vec_sim)\n        else:\n            fused = 0.58 * local_score + 0.42 * max(0.0, vec_sim)\n    else:\n        fused = 0.48 * local_score + 0.52 * max(0.0, vec_sim)\n    fused *= context_factor\n    return {\n        \"vec_sim\": vec_sim,\n        \"local_score\": float(local_score),\n        \"fused_score\": float(fused),\n        \"good_matches\": int(len(good)),\n        \"inliers\": int(inliers),\n        \"inlier_ratio\": float(inlier_ratio),\n        \"context_factor\": context_factor,\n        \"context_relation\": context_relation,\n        \"orientation_a\": a.orientation,\n        \"orientation_b\": b.orientation,\n        \"body_kind_a\": a.body_kind,\n        \"body_kind_b\": b.body_kind,\n        \"low_light_a\": bool(a.low_light),\n        \"low_light_b\": bool(b.low_light),\n    }\n\n\ndef sample_train_rows(rows: pd.DataFrame, max_images: int) -> pd.DataFrame:\n    train = rows[(rows[\"split\"].eq(\"train\")) & rows[\"identity\"].astype(str).ne(\"\")].copy()\n    if max_images <= 0 or len(train) <= max_images:\n        return train\n    # Keep identity diversity; no single individual should dominate calibration.\n    parts = []\n    for _, group in train.groupby(\"identity\", sort=False):\n        parts.append(group.head(6))\n    sampled = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED)\n    return sampled.head(max_images).copy()\n\n\ndef calibration_pairs(items: list[FeatureItem], top_k: int, budget: int) -> list[tuple[int, int, float]]:\n    by_id = {it.image_id: it for it in items}\n    positives = []\n    groups: dict[str, list[int]] = {}\n    for it in items:\n        if it.identity:\n            groups.setdefault(it.identity, []).append(it.image_id)\n    for members in groups.values():\n        if len(members) < 2:\n            continue\n        combos = [\n            (a, b)\n            for a, b in itertools.combinations(sorted(members), 2)\n            if orientation_compatible(by_id[a], by_id[b])\n        ]\n        positives.extend(combos[: min(len(combos), 12)])\n    pairs = cosine_pairs(items, top_k=top_k, budget=max(budget, len(positives) * 3))\n    out: dict[tuple[int, int], float] = {}\n    for a, b in positives:\n        out[(a, b)] = float(np.dot(by_id[a].vec, by_id[b].vec))\n    for a, b, sim in pairs:\n        out[(a, b)] = sim\n        if len(out) >= budget:\n            break\n    return [(a, b, s) for (a, b), s in out.items()]\n\n\ndef score_pairs(items: list[FeatureItem], pairs: list[tuple[int, int, float]], detector_name: str) -> pd.DataFrame:\n    by_id = {it.image_id: it for it in items}\n    rows = []\n    for idx, (a_id, b_id, prior_sim) in enumerate(pairs, start=1):\n        a = by_id.get(int(a_id))\n        b = by_id.get(int(b_id))\n        if a is None or b is None:\n            continue\n        s = score_pair(a, b, detector_name)\n        same_identity = bool(a.identity and a.identity == b.identity)\n        same_cluster = bool(a.cluster and a.cluster == b.cluster)\n        rows.append(\n            {\n                \"species\": a.species,\n                \"image_id_a\": int(a_id),\n                \"image_id_b\": int(b_id),\n                \"same_identity\": same_identity,\n                \"same_base_cluster\": same_cluster,\n                \"prior_vec_sim\": float(prior_sim),\n                \"cluster_a\": a.cluster,\n                \"cluster_b\": b.cluster,\n                **s,\n            }\n        )\n        if idx % 10000 == 0:\n            print(f\"[{a.species}] scored {idx}/{len(pairs)} pairs\")\n    return pd.DataFrame(rows)\n\n\ndef threshold_for_precision(cal: pd.DataFrame, target_precision: float, species: str) -> dict:\n    if cal.empty or \"same_identity\" not in cal.columns:\n        return default_threshold(species, target_precision)\n    y = cal[\"same_identity\"].astype(bool).to_numpy()\n    scores = cal[\"fused_score\"].astype(float).to_numpy()\n    if y.sum() == 0 or (~y).sum() == 0:\n        return default_threshold(species, target_precision)\n    order = np.argsort(-scores)\n    y_sorted = y[order]\n    s_sorted = scores[order]\n    tp = np.cumsum(y_sorted)\n    fp = np.cumsum(~y_sorted)\n    precision = tp / np.maximum(1, tp + fp)\n    recall = tp / max(1, int(y.sum()))\n    candidates = np.where((precision >= target_precision) & ((tp + fp) >= 3))[0]\n    if len(candidates):\n        idx = int(candidates[-1])\n    else:\n        # Bolder fallback: use the positive median but report that precision was not met.\n        pos_scores = scores[y]\n        idx = int(np.argmin(np.abs(s_sorted - float(np.percentile(pos_scores, 55)))))\n    thr = float(s_sorted[idx])\n    accepted = scores >= thr\n    row = {\n        \"threshold\": thr,\n        \"target_precision\": float(target_precision),\n        \"cal_precision\": float(((accepted & y).sum()) / max(1, int(accepted.sum()))),\n        \"cal_recall\": float(((accepted & y).sum()) / max(1, int(y.sum()))),\n        \"cal_accepted\": int(accepted.sum()),\n        \"cal_positive\": int(y.sum()),\n        \"cal_negative\": int((~y).sum()),\n        \"cal_auc_rank\": auc_rank(y, scores),\n    }\n    return row\n\n\ndef default_threshold(species: str, target_precision: float) -> dict:\n    base = {\"LynxID2025\": 0.62, \"SalamanderID2025\": 0.58, \"SeaTurtleID2022\": 0.66}.get(species, 0.62)\n    return {\n        \"threshold\": base,\n        \"target_precision\": target_precision,\n        \"cal_precision\": 0.0,\n        \"cal_recall\": 0.0,\n        \"cal_accepted\": 0,\n        \"cal_positive\": 0,\n        \"cal_negative\": 0,\n        \"cal_auc_rank\": 0.5,\n    }\n\n\ndef auc_rank(y_true: np.ndarray, scores: np.ndarray) -> float:\n    y = y_true.astype(bool)\n    n_pos = int(y.sum())\n    n_neg = int((~y).sum())\n    if n_pos == 0 or n_neg == 0:\n        return 0.5\n    order = np.argsort(scores)\n    ranks = np.empty_like(order, dtype=np.float64)\n    ranks[order] = np.arange(1, len(scores) + 1)\n    pos_ranks = ranks[y].sum()\n    return float((pos_ranks - n_pos * (n_pos + 1) / 2) / max(1, n_pos * n_neg))\n\n\ndef labels_for_species(submission: pd.DataFrame, rows: pd.DataFrame) -> dict[int, str]:\n    merged = rows[[\"image_id\"]].merge(submission, on=\"image_id\", how=\"left\")\n    return dict(zip(merged[\"image_id\"].astype(int), merged[\"cluster\"].astype(str)))\n\n\ndef relabel(ids: list[int], uf: UnionFind, species: str, variant: str) -> dict[int, str]:\n    order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(ids):\n        root = uf.find(image_id)\n        if root not in order:\n            order[root] = len(order)\n        labels[image_id] = f\"cluster_{species}_{order[root]}\"\n    return labels\n\n\nPROFILE = {\n    \"independent_strict\": {\n        \"precision\": {\"LynxID2025\": 0.82, \"SalamanderID2025\": 0.78, \"SeaTurtleID2022\": 0.84},\n        \"thr_scale\": 0.98,\n        \"min_inliers\": {\"LynxID2025\": 6, \"SalamanderID2025\": 5, \"SeaTurtleID2022\": 6},\n        \"max_edges\": {\"LynxID2025\": 520, \"SalamanderID2025\": 150, \"SeaTurtleID2022\": 180},\n    },\n    \"independent_balanced\": {\n        \"precision\": {\"LynxID2025\": 0.66, \"SalamanderID2025\": 0.62, \"SeaTurtleID2022\": 0.70},\n        \"thr_scale\": 0.86,\n        \"min_inliers\": {\"LynxID2025\": 4, \"SalamanderID2025\": 3, \"SeaTurtleID2022\": 4},\n        \"max_edges\": {\"LynxID2025\": 820, \"SalamanderID2025\": 360, \"SeaTurtleID2022\": 340},\n    },\n    \"independent_swing\": {\n        \"precision\": {\"LynxID2025\": 0.52, \"SalamanderID2025\": 0.50, \"SeaTurtleID2022\": 0.58},\n        \"thr_scale\": 0.74,\n        \"min_inliers\": {\"LynxID2025\": 3, \"SalamanderID2025\": 2, \"SeaTurtleID2022\": 3},\n        \"max_edges\": {\"LynxID2025\": 900, \"SalamanderID2025\": 760, \"SeaTurtleID2022\": 430},\n    },\n    \"independent_wild\": {\n        \"precision\": {\"LynxID2025\": 0.42, \"SalamanderID2025\": 0.40, \"SeaTurtleID2022\": 0.48},\n        \"thr_scale\": 0.62,\n        \"min_inliers\": {\"LynxID2025\": 2, \"SalamanderID2025\": 1, \"SeaTurtleID2022\": 2},\n        \"max_edges\": {\"LynxID2025\": 930, \"SalamanderID2025\": 1300, \"SeaTurtleID2022\": 460},\n    },\n}\n\n\nGRAPH_GUARD = {\n    \"independent_strict\": {\n        # Component caps are tied to train identity-size priors:\n        # Lynx max/p99/p95 = 353/289/151, Salamander = 12/10/7,\n        # SeaTurtle = 190/98/55. Strict uses roughly p95, wild uses max.\n        \"max_component\": {\"LynxID2025\": 151, \"SalamanderID2025\": 7, \"SeaTurtleID2022\": 55},\n        \"max_degree\": {\"LynxID2025\": 3, \"SalamanderID2025\": 2, \"SeaTurtleID2022\": 2},\n        \"mutual_rank\": {\"LynxID2025\": 8, \"SalamanderID2025\": 3, \"SeaTurtleID2022\": 4},\n    },\n    \"independent_balanced\": {\n        \"max_component\": {\"LynxID2025\": 289, \"SalamanderID2025\": 10, \"SeaTurtleID2022\": 98},\n        \"max_degree\": {\"LynxID2025\": 4, \"SalamanderID2025\": 2, \"SeaTurtleID2022\": 3},\n        \"mutual_rank\": {\"LynxID2025\": 10, \"SalamanderID2025\": 3, \"SeaTurtleID2022\": 5},\n    },\n    \"independent_swing\": {\n        \"max_component\": {\"LynxID2025\": 353, \"SalamanderID2025\": 12, \"SeaTurtleID2022\": 190},\n        \"max_degree\": {\"LynxID2025\": 5, \"SalamanderID2025\": 2, \"SeaTurtleID2022\": 3},\n        \"mutual_rank\": {\"LynxID2025\": 12, \"SalamanderID2025\": 3, \"SeaTurtleID2022\": 6},\n    },\n    \"independent_wild\": {\n        \"max_component\": {\"LynxID2025\": 353, \"SalamanderID2025\": 12, \"SeaTurtleID2022\": 190},\n        \"max_degree\": {\"LynxID2025\": 6, \"SalamanderID2025\": 3, \"SeaTurtleID2022\": 4},\n        \"mutual_rank\": {\"LynxID2025\": 15, \"SalamanderID2025\": 4, \"SeaTurtleID2022\": 7},\n    },\n}\n\n\ndef edge_rank_maps(pair_scores: pd.DataFrame) -> dict[tuple[int, int], int]:\n    neighbors: dict[int, list[tuple[int, float]]] = {}\n    for row in pair_scores.itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        s = float(row.fused_score)\n        neighbors.setdefault(a, []).append((b, s))\n        neighbors.setdefault(b, []).append((a, s))\n    ranks: dict[tuple[int, int], int] = {}\n    for node, vals in neighbors.items():\n        vals.sort(key=lambda x: -x[1])\n        for rank, (other, _) in enumerate(vals, start=1):\n            ranks[(node, other)] = rank\n    return ranks\n\n\ndef uf_members(uf: UnionFind, ids: list[int], root: int) -> list[int]:\n    return [image_id for image_id in ids if uf.find(image_id) == root]\n\n\ndef salamander_edge_allowed(row, threshold: float, min_inliers: int) -> bool:\n    a_kind = str(getattr(row, \"body_kind_a\", \"body\"))\n    b_kind = str(getattr(row, \"body_kind_b\", \"body\"))\n    if a_kind == b_kind:\n        return True\n    # Head-only to full-body matches can be real, but they should not be the\n    # easy bridge that percolates all salamanders into one component.\n    return (\n        str(getattr(row, \"orientation_a\", \"\")) == str(getattr(row, \"orientation_b\", \"\"))\n        and int(getattr(row, \"inliers\", 0)) >= max(8, min_inliers + 4)\n        and float(getattr(row, \"fused_score\", 0.0)) >= threshold * 1.18\n    )\n\n\ndef graph_guard_for(species: str, variant: str) -> dict[str, int]:\n    guard = GRAPH_GUARD.get(variant, GRAPH_GUARD[\"independent_strict\"])\n    return {\n        \"max_component\": int(guard[\"max_component\"].get(species, 8)),\n        \"max_degree\": int(guard[\"max_degree\"].get(species, 2)),\n        \"mutual_rank\": int(guard[\"mutual_rank\"].get(species, 3)),\n    }\n\n\ndef graph_merge_species(\n    species: str,\n    rows: pd.DataFrame,\n    seed_labels: dict[int, str],\n    pair_scores: pd.DataFrame,\n    threshold: float,\n    min_inliers: int,\n    max_edges: int,\n    variant: str,\n    preserve_existing: bool = False,\n) -> tuple[dict[int, str], list[dict]]:\n    ids = rows[\"image_id\"].astype(int).tolist()\n    uf = UnionFind(ids)\n    if preserve_existing:\n        by_cluster: dict[str, list[int]] = {}\n        for image_id in ids:\n            by_cluster.setdefault(seed_labels[image_id], []).append(image_id)\n        for members in by_cluster.values():\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n    accepted: list[dict] = []\n    if pair_scores.empty:\n        return relabel(ids, uf, species, variant), accepted\n    cross = pair_scores[~pair_scores[\"same_base_cluster\"].astype(bool)].copy()\n    if cross.empty:\n        return relabel(ids, uf, species, variant), accepted\n    guard = graph_guard_for(species, variant)\n    ranks = edge_rank_maps(cross)\n    node_degree = {image_id: 0 for image_id in ids}\n    cross = cross.sort_values([\"fused_score\", \"inliers\", \"vec_sim\"], ascending=[False, False, False])\n    for row in cross.itertuples(index=False):\n        if len(accepted) >= max_edges:\n            break\n        if float(row.fused_score) < threshold:\n            continue\n        if int(row.inliers) < min_inliers and float(row.local_score) < max(0.44, threshold * 0.74):\n            continue\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        rank_a = int(ranks.get((a, b), 999))\n        rank_b = int(ranks.get((b, a), 999))\n        if rank_a > guard[\"mutual_rank\"] or rank_b > guard[\"mutual_rank\"]:\n            continue\n        if node_degree.get(a, 0) >= guard[\"max_degree\"] or node_degree.get(b, 0) >= guard[\"max_degree\"]:\n            continue\n        if species == \"SalamanderID2025\" and not salamander_edge_allowed(row, threshold, min_inliers):\n            continue\n        ra = uf.find(a)\n        rb = uf.find(b)\n        if ra == rb:\n            continue\n        if uf.size[ra] + uf.size[rb] > guard[\"max_component\"]:\n            continue\n        if uf.union(a, b):\n            node_degree[a] = node_degree.get(a, 0) + 1\n            node_degree[b] = node_degree.get(b, 0) + 1\n            accepted.append(\n                {\n                    \"variant\": variant,\n                    \"species\": species,\n                    \"guard_max_component\": int(guard[\"max_component\"]),\n                    \"guard_max_degree\": int(guard[\"max_degree\"]),\n                    \"guard_mutual_rank\": int(guard[\"mutual_rank\"]),\n                    \"rank_a_to_b\": rank_a,\n                    \"rank_b_to_a\": rank_b,\n                    \"image_id_a\": a,\n                    \"image_id_b\": b,\n                    \"fused_score\": float(row.fused_score),\n                    \"vec_sim\": float(row.vec_sim),\n                    \"local_score\": float(row.local_score),\n                    \"inliers\": int(row.inliers),\n                    \"good_matches\": int(row.good_matches),\n                    \"cluster_a\": str(row.cluster_a),\n                    \"cluster_b\": str(row.cluster_b),\n                    \"context_relation\": str(getattr(row, \"context_relation\", \"\")),\n                    \"orientation_a\": str(getattr(row, \"orientation_a\", \"\")),\n                    \"orientation_b\": str(getattr(row, \"orientation_b\", \"\")),\n                    \"body_kind_a\": str(getattr(row, \"body_kind_a\", \"\")),\n                    \"body_kind_b\": str(getattr(row, \"body_kind_b\", \"\")),\n                }\n            )\n    return relabel(ids, uf, species, variant), accepted\n\n\nSHAPE_TARGETS = {\n    # These are not seed labels. They are sanity targets for open-set cluster\n    # shape, based on train identity multiplicities plus the best submitted\n    # non-collapsed species profile. The old chooser only blocked huge clusters,\n    # which let it select submissions that were mathematically over-split.\n    \"LynxID2025\": {\n        \"clusters\": 135,\n        \"min_clusters\": 75,\n        \"max_clusters\": 260,\n        \"max_cluster\": 67,\n        \"min_max_cluster\": 28,\n        \"max_max_cluster\": 125,\n        \"singletons\": 32,\n        \"max_singletons\": 320,\n    },\n    \"SalamanderID2025\": {\n        \"clusters\": 459,\n        \"min_clusters\": 380,\n        \"max_clusters\": 560,\n        \"max_cluster\": 9,\n        \"min_max_cluster\": 4,\n        \"max_max_cluster\": 16,\n        \"singletons\": 330,\n        \"max_singletons\": 450,\n    },\n    \"SeaTurtleID2022\": {\n        \"clusters\": 166,\n        \"min_clusters\": 95,\n        \"max_clusters\": 290,\n        \"max_cluster\": 13,\n        \"min_max_cluster\": 6,\n        \"max_max_cluster\": 42,\n        \"singletons\": 60,\n        \"max_singletons\": 220,\n    },\n    \"TexasHornedLizards\": {\n        \"clusters\": 74,\n        \"min_clusters\": 35,\n        \"max_clusters\": 155,\n        \"max_cluster\": 26,\n        \"min_max_cluster\": 8,\n        \"max_max_cluster\": 55,\n        \"singletons\": 26,\n        \"max_singletons\": 145,\n    },\n}\n\nVARIANT_BIAS = {\n    \"independent_swing\": -0.04,\n    \"independent_wild\": 0.12,\n}\n\n\ndef log_ratio(value: float, target: float) -> float:\n    return abs(math.log(max(1.0, value + 1.0) / max(1.0, target + 1.0)))\n\n\ndef species_shape_score(row) -> tuple[float, list[str]]:\n    species = str(row.species)\n    target = SHAPE_TARGETS.get(species)\n    if target is None:\n        return 0.0, []\n    n_clusters = int(row.n_clusters)\n    singletons = int(row.singletons)\n    max_cluster = int(row.max_cluster)\n    notes: list[str] = []\n    if n_clusters <= 1 and int(row.n_images) > 1:\n        notes.append(\"collapsed\")\n        return 999.0, notes\n\n    score = 2.6 * log_ratio(n_clusters, target[\"clusters\"])\n    score += 1.4 * log_ratio(max_cluster, target[\"max_cluster\"])\n    score += 0.55 * log_ratio(singletons, target[\"singletons\"])\n\n    if n_clusters < target[\"min_clusters\"]:\n        score += 3.0 * log_ratio(n_clusters, target[\"min_clusters\"])\n        notes.append(\"too_few_clusters\")\n    if n_clusters > target[\"max_clusters\"]:\n        score += 3.0 * log_ratio(n_clusters, target[\"max_clusters\"])\n        notes.append(\"too_many_clusters\")\n    if max_cluster < target[\"min_max_cluster\"]:\n        score += 1.6 * log_ratio(max_cluster, target[\"min_max_cluster\"])\n        notes.append(\"max_cluster_too_small\")\n    if max_cluster > target[\"max_max_cluster\"]:\n        score += 12.0 * log_ratio(max_cluster, target[\"max_max_cluster\"])\n        notes.append(\"max_cluster_too_large\")\n    if singletons > target[\"max_singletons\"]:\n        score += 1.5 * log_ratio(singletons, target[\"max_singletons\"])\n        notes.append(\"too_many_singletons\")\n    return float(score), notes\n\n\ndef candidate_shape_scores(candidate_report: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for variant, part in candidate_report.groupby(\"variant\", sort=False):\n        total = float(VARIANT_BIAS.get(str(variant), 0.0))\n        notes = []\n        for row in part.itertuples(index=False):\n            score, species_notes = species_shape_score(row)\n            total += score\n            if species_notes:\n                notes.append(f\"{row.species}:{'|'.join(species_notes)}\")\n        rows.append(\n            {\n                \"variant\": str(variant),\n                \"shape_score\": float(total),\n                \"notes\": \";\".join(notes),\n            }\n        )\n    out = pd.DataFrame(rows)\n    if not out.empty:\n        out = out.sort_values([\"shape_score\", \"variant\"], ascending=[True, True]).reset_index(drop=True)\n    return out\n\n\ndef choose_recommended_submission(candidate_report: pd.DataFrame, written: dict[str, str]) -> tuple[str, Path, pd.DataFrame]:\n    shape_scores = candidate_shape_scores(candidate_report)\n    for row in shape_scores.itertuples(index=False):\n        variant = str(row.variant)\n        if variant in written:\n            return variant, Path(written[variant]), shape_scores\n    variant, path = next(iter(written.items()))\n    return variant, Path(path), shape_scores\n\n\ndef compact_submission_labels(sub: pd.DataFrame, view_df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Normalize final labels to cluster_<SpeciesID>_<integer>.\n\n    Kaggle only needs stable cluster identifiers. The algorithm's internal\n    variant/version names are useful in reports, but long labels are risky and\n    ugly in the actual upload file.\n    \"\"\"\n    species_by_id = dict(zip(view_df[\"image_id\"].astype(int), view_df[\"species_id\"].astype(str)))\n    work = sub.copy()\n    work[\"image_id\"] = work[\"image_id\"].astype(int)\n    work[\"_species_id\"] = work[\"image_id\"].map(species_by_id).fillna(\"AnimalCLEF\").astype(str)\n    compact = {}\n    for species in SPECIES_ORDER:\n        mask = work[\"_species_id\"].eq(species)\n        labels = work.loc[mask, \"cluster\"].astype(str)\n        order = {old: f\"cluster_{species}_{idx}\" for idx, old in enumerate(sorted(labels.unique()))}\n        compact.update(order)\n    other = work.loc[~work[\"_species_id\"].isin(SPECIES_ORDER), \"cluster\"].astype(str)\n    for idx, old in enumerate(sorted(other.unique())):\n        compact.setdefault(old, f\"cluster_AnimalCLEF_{idx}\")\n    work[\"cluster\"] = work[\"cluster\"].astype(str).map(compact)\n    return work[[\"image_id\", \"cluster\"]]\n\n\ndef build_submission(sample: pd.DataFrame, view_df: pd.DataFrame, updates: dict[int, str], variant: str, out_path: Path) -> pd.DataFrame:\n    sub = sample[[\"image_id\"]].copy()\n    species_by_id = dict(zip(view_df[\"image_id\"].astype(int), view_df[\"species_id\"].astype(str)))\n    def label(image_id: int) -> str:\n        if image_id in updates:\n            return updates[image_id]\n        species = species_by_id.get(image_id, \"AnimalCLEF\")\n        return f\"cluster_{species}_singleton_{image_id}\"\n\n    sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(label)\n    sub = compact_submission_labels(sub, view_df)\n    validate_submission(sub, sample)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    sub.to_csv(out_path, index=False)\n    return sub\n\n\ndef write_species_fragments(sub: pd.DataFrame, view_df: pd.DataFrame, variant: str, fragments_dir: Path) -> None:\n    \"\"\"Write one replaceable cluster fragment per species.\n\n    This keeps the final notebook usable as a one-shot submission generator,\n    while also giving us a clean contract for parallel species notebooks:\n    produce image_id/cluster rows for one species, then a later merger can\n    splice that fragment without touching the other species.\n    \"\"\"\n    species_by_id = dict(zip(view_df[\"image_id\"].astype(int), view_df[\"species_id\"].astype(str)))\n    work = sub.copy()\n    work[\"image_id\"] = work[\"image_id\"].astype(int)\n    work[\"species_id\"] = work[\"image_id\"].map(species_by_id).fillna(\"AnimalCLEF\").astype(str)\n    fragments_dir.mkdir(parents=True, exist_ok=True)\n    for species in SPECIES_ORDER:\n        frag = work[work[\"species_id\"].eq(species)][[\"image_id\", \"cluster\", \"species_id\"]].copy()\n        frag.insert(2, \"variant\", variant)\n        frag[\"source_version\"] = VERSION\n        frag.to_csv(fragments_dir / f\"fragment_{VERSION}_{variant}_{species}.csv\", index=False)\n\n\ndef summarize_submission(sub: pd.DataFrame, reference: pd.DataFrame | None, view_df: pd.DataFrame, variant: str) -> list[dict]:\n    sub_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    ref_map = dict(zip(reference[\"image_id\"].astype(int), reference[\"cluster\"].astype(str))) if reference is not None else {}\n    rows = []\n    for species in SPECIES_ORDER:\n        ids = view_df.loc[(view_df[\"split\"].eq(\"test\")) & (view_df[\"species_id\"].eq(species)), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        changed = sum(1 for i in ids if ref_map and sub_map[i] != ref_map.get(i))\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": int(len(ids)),\n                \"n_clusters\": int(counts.shape[0]),\n                \"singletons\": int((counts == 1).sum()),\n                \"max_cluster\": int(counts.max()) if len(counts) else 0,\n                \"membership_changed_vs_optional_reference\": int(changed) if ref_map else -1,\n            }\n        )\n    return rows\n\n\ndef import_texas_helpers():\n    try:\n        import texas_astrodot_2025reuse_v20260426 as texas_base\n        import texas_astrodot_nooval_v20260427 as texas_nooval\n\n        # Make the Texas rule explicit at the final fusion layer too. The\n        # field photos are ventral views with the head already at the top, so\n        # the base helper must never vertically flip the belly-dot template.\n        if hasattr(texas_nooval, \"align_vertical_no_flip\"):\n            texas_base.align_vertical = texas_nooval.align_vertical_no_flip\n        if hasattr(texas_nooval, \"texas_belly_template_no_oval\"):\n            texas_base.texas_belly_template = texas_nooval.texas_belly_template_no_oval\n        if hasattr(texas_nooval, \"texas_pair_score_no_flip\"):\n            texas_base.texas_pair_score = texas_nooval.texas_pair_score_no_flip\n        return texas_base, texas_nooval\n    except Exception as exc:\n        print(f\"[Texas] helper import failed: {exc}\")\n        return None, None\n\n\ndef estimate_non_grey_foreground(rgb: np.ndarray) -> np.ndarray:\n    diff_grey = np.abs(rgb.astype(np.int16) - NEUTRAL_GREY_RGB.reshape(1, 1, 3).astype(np.int16)).sum(axis=2)\n    diff_black = np.abs(rgb.astype(np.int16) - BLACK_RGB.reshape(1, 1, 3).astype(np.int16)).sum(axis=2)\n    mask = ((diff_grey > 24) & (diff_black > 18)).astype(np.uint8) * 255\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n > 1:\n        biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n        mask = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.02 or coverage > 0.94:\n        mask = np.full(mask.shape[:2], 255, dtype=np.uint8)\n    return mask\n\n\ndef write_neutral_grey_view(rgb: np.ndarray, mask: np.ndarray, out_img: Path, out_mask: Path) -> None:\n    out_img.parent.mkdir(parents=True, exist_ok=True)\n    out_mask.parent.mkdir(parents=True, exist_ok=True)\n    prepared = np.where(mask[:, :, None] > 0, rgb, NEUTRAL_GREY_RGB.reshape(1, 1, 3)).astype(np.uint8)\n    cv2.imwrite(str(out_img), cv2.cvtColor(prepared, cv2.COLOR_RGB2BGR))\n    cv2.imwrite(str(out_mask), mask)\n\n\ndef texas_labels_from_views(\n    rows: pd.DataFrame,\n    base_labels: dict[int, str],\n    args: argparse.Namespace,\n    reports_dir: Path,\n    viz_dir: Path | None,\n) -> dict[str, dict[int, str]]:\n    texas_base, texas_nooval = import_texas_helpers()\n    if texas_base is None or texas_nooval is None or rows.empty:\n        return {}\n    helper_args = argparse.Namespace(max_side=args.max_side, texas_canvas_w=224, texas_canvas_h=320)\n    prepared_dir = reports_dir.parent / \"prepared_views\" / \"TexasHornedLizards\"\n    items = []\n    for idx, row in enumerate(rows.to_dict(\"records\"), start=1):\n        rec = dict(row)\n        original_path = str(rec.get(\"original_path\", \"\") or \"\")\n        final_path = str(\n            rec.get(\"view_species_final_path\")\n            or rec.get(\"view_sam_yolo_union_path\")\n            or rec.get(\"view_sam_clean_path\")\n            or original_path\n        )\n        # Final correction: Texas uses the SAM+YOLO final/union crop as the\n        # image source. No oval is applied and the no-flip monkey patch above\n        # keeps the head-at-top acquisition protocol intact. A neutral-grey\n        # background is written before the astro-dot template is extracted.\n        rec[\"source_path\"] = original_path\n        rec[\"view_path\"] = final_path\n        rec[\"mask_path\"] = \"\"\n        rec[\"view_source\"] = \"sam_yolo_final_neutral_grey_nooval_noflip\"\n        if final_path and Path(final_path).exists():\n            try:\n                rgb = read_rgb(final_path, args.max_side)\n                mask = estimate_non_grey_foreground(rgb)\n                out_img = prepared_dir / f\"{TEXAS}_{int(row['image_id'])}_neutralgrey.jpg\"\n                out_mask = prepared_dir / f\"{TEXAS}_{int(row['image_id'])}_neutralgrey_mask.png\"\n                write_neutral_grey_view(rgb, mask, out_img, out_mask)\n                rec[\"view_path\"] = str(out_img)\n                rec[\"mask_path\"] = str(out_mask)\n            except Exception as exc:\n                print(f\"[Texas] neutral-grey prep failed image_id={row.get('image_id')}: {exc}\")\n        try:\n            items.append(texas_nooval.texas_belly_template_no_oval(rec, base_labels[int(row[\"image_id\"])], helper_args))\n        except Exception as exc:\n            print(f\"[Texas] template failed image_id={row.get('image_id')}: {exc}\")\n        if idx % 75 == 0:\n            print(f\"[Texas no-oval YOLO-view] templates {idx}/{len(rows)}\")\n    if not items:\n        return {}\n    pair_scores = texas_base.score_all_texas_pairs(items, args.texas_pair_budget)\n    pair_scores.to_csv(reports_dir / f\"{VERSION}_Texas_view_nooval_pair_scores.csv\", index=False)\n    pd.DataFrame(\n        [\n            {\n                \"image_id\": item.image_id,\n                \"view_path\": item.view_path,\n                \"view_source\": item.view_source,\n                \"quality\": item.quality,\n                \"dot_points\": len(item.dot_points),\n                \"mask_coverage\": item.debug.get(\"mask_coverage\", np.nan),\n                \"pca_angle\": item.debug.get(\"pca_angle\", np.nan),\n                \"rotate_angle\": item.debug.get(\"rotate_angle\", np.nan),\n                \"flipped_vertical\": item.debug.get(\"flipped_vertical\", False),\n                \"orientation_rule\": item.debug.get(\"orientation_rule\", \"head_already_top_no_flip\"),\n                \"mask_mode\": item.debug.get(\"mask_mode\", \"full_aligned_crop_no_oval\"),\n            }\n            for item in items\n        ]\n    ).to_csv(reports_dir / f\"{VERSION}_Texas_template_report.csv\", index=False)\n    labels = {}\n    for variant in [\"merge_ultra\", \"splitmerge_guarded\", \"splitmerge_swing\"]:\n        try:\n            labels[variant] = texas_independent_variant_labels(items, pair_scores, variant)\n        except Exception as exc:\n            print(f\"[Texas] variant {variant} failed: {exc}\")\n    if viz_dir is not None:\n        try:\n            texas_base.save_texas_preview(items, viz_dir / f\"{VERSION}_Texas_nooval_yoloview_preview.jpg\", args.visual_limit)\n            texas_base.save_pair_preview(items, pair_scores, viz_dir / f\"{VERSION}_Texas_nooval_yoloview_top_pairs.jpg\", max(6, args.visual_limit // 2), True)\n        except Exception as exc:\n            print(f\"[Texas] visualization failed: {exc}\")\n    return labels\n\n\ndef texas_independent_variant_labels(items: list, pair_scores: pd.DataFrame, variant: str) -> dict[int, str]:\n    \"\"\"Texas clustering without any previous submission seed.\n\n    Texas has no train identities, but the acquisition protocol is unusually\n    standardized: ventral belly, head at top. This uses mutual high-rank dot\n    matches as graph edges and caps component growth to avoid one large dot\n    texture attractor.\n    \"\"\"\n    ids = [int(it.image_id) for it in items]\n    uf = UnionFind(ids)\n    if pair_scores.empty:\n        return relabel(ids, uf, TEXAS, f\"{variant}_texas_independent\")\n    if variant == \"merge_ultra\":\n        thr, rank_limit, max_comp, max_edges = 0.565, 2, 12, 95\n        min_overlap, min_point, min_stack = 0.42, 0.46, 0.500\n    elif variant == \"splitmerge_guarded\":\n        thr, rank_limit, max_comp, max_edges = 0.540, 4, 24, 190\n        min_overlap, min_point, min_stack = 0.35, 0.42, 0.490\n    else:\n        thr, rank_limit, max_comp, max_edges = 0.518, 6, 36, 235\n        min_overlap, min_point, min_stack = 0.30, 0.39, 0.480\n\n    cand = pair_scores[\n        (pair_scores[\"score\"].astype(float) >= thr)\n        & (pair_scores[\"overlap\"].astype(float) >= min_overlap)\n        & (pair_scores[\"point_score\"].astype(float) >= min_point)\n        & (pair_scores[\"stack_gain\"].astype(float) >= min_stack)\n        & (pair_scores[\"transform\"].astype(str).eq(\"identity\"))\n    ].copy()\n    if cand.empty:\n        return relabel(ids, uf, TEXAS, f\"{variant}_texas_independent\")\n\n    neighbors: dict[int, list[tuple[int, float]]] = {i: [] for i in ids}\n    for row in cand.itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        s = float(row.score)\n        neighbors[a].append((b, s))\n        neighbors[b].append((a, s))\n    ranks: dict[tuple[int, int], int] = {}\n    for node, vals in neighbors.items():\n        vals.sort(key=lambda x: -x[1])\n        for rank, (other, _) in enumerate(vals, start=1):\n            ranks[(node, other)] = rank\n\n    accepted = 0\n    for row in cand.sort_values([\"score\", \"overlap\", \"point_score\"], ascending=False).itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        if ranks.get((a, b), 999) > rank_limit or ranks.get((b, a), 999) > rank_limit:\n            continue\n        ra = uf.find(a)\n        rb = uf.find(b)\n        if ra == rb:\n            continue\n        if uf.size[ra] + uf.size[rb] > max_comp:\n            continue\n        if uf.union(a, b):\n            accepted += 1\n            if accepted >= max_edges:\n                break\n    return relabel(ids, uf, TEXAS, f\"{variant}_texas_independent\")\n\n\ndef thumb(path: str, size: tuple[int, int]) -> Image.Image:\n    try:\n        img = Image.open(path).convert(\"RGB\")\n    except Exception:\n        img = Image.new(\"RGB\", size, (20, 20, 20))\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (18, 18, 18))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef save_edge_preview(view_df: pd.DataFrame, accepted: pd.DataFrame, out_path: Path, limit: int) -> None:\n    if accepted.empty:\n        return\n    row_by_id = {int(r[\"image_id\"]): r for r in view_df.to_dict(\"records\")}\n    edges = accepted.sort_values(\"fused_score\", ascending=False).head(limit)\n    tile = (230, 180)\n    label_h = 36\n    canvas = Image.new(\"RGB\", (tile[0] * 2, len(edges) * (tile[1] + label_h)), (18, 18, 18))\n    draw = ImageDraw.Draw(canvas)\n    for ridx, edge in enumerate(edges.to_dict(\"records\")):\n        y = ridx * (tile[1] + label_h)\n        text = (\n            f\"{edge['variant']} {edge['species']} {edge['image_id_a']} vs {edge['image_id_b']} \"\n            f\"s={float(edge['fused_score']):.3f} inl={int(edge.get('inliers', 0))}\"\n        )\n        draw.text((5, y + 4), text[:90], fill=(255, 240, 140))\n        for c, image_id in enumerate([int(edge[\"image_id_a\"]), int(edge[\"image_id_b\"])]):\n            row = row_by_id.get(image_id, {})\n            p = str(row.get(\"view_species_final_path\") or row.get(\"view_sam_yolo_union_path\") or row.get(\"original_path\") or \"\")\n            canvas.paste(thumb(p, tile), (c * tile[0], y + label_h))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.smoke:\n        args.train_images_per_species = min(args.train_images_per_species, 80)\n        args.train_pairs_per_species = min(args.train_pairs_per_species, 240)\n        args.pair_budget_per_species = min(args.pair_budget_per_species, 900)\n        args.texas_pair_budget = min(args.texas_pair_budget, 900)\n        args.top_k = min(args.top_k, 16)\n    random.seed(SEED)\n    np.random.seed(SEED)\n\n    output_root = Path(args.output_root)\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    fragments_dir = output_root / \"species_fragments\"\n    viz_dir = output_root / \"visualizations\" if args.save_visualizations else None\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n    fragments_dir.mkdir(parents=True, exist_ok=True)\n    if viz_dir is not None:\n        viz_dir.mkdir(parents=True, exist_ok=True)\n\n    data_root = find_data_root(args.data_root)\n    view_manifest = find_view_manifest(args.view_manifest)\n    view_df = prepare_view_manifest(view_manifest, data_root)\n    sample = pd.read_csv(data_root / \"sample_submission.csv\")\n    reference_arg = args.reference_submission or args.base_submission\n    reference_path = find_reference_submission(reference_arg, data_root)\n    reference = load_submission(reference_path) if reference_path else None\n    if reference is not None:\n        validate_submission(reference, sample)\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"view_manifest={view_manifest}\")\n    print(f\"reference_submission={reference_path}\")\n    print(f\"output_root={output_root}\")\n\n    detector_name, detector = make_detector()\n    print(f\"detector={detector_name}\")\n\n    test_rows_by_species = {\n        species: view_df[(view_df[\"split\"].eq(\"test\")) & (view_df[\"species_id\"].eq(species))].sort_values(\"image_id\").copy()\n        for species in SPECIES_ORDER\n    }\n    source_rows = []\n    cal_thresholds: dict[str, dict[str, dict]] = {}\n    test_pair_scores: dict[str, pd.DataFrame] = {}\n    item_cache: dict[str, list[FeatureItem]] = {}\n    feature_rows: list[dict] = []\n\n    for species in REID_SPECIES:\n        test_rows = test_rows_by_species[species]\n        if args.smoke:\n            test_rows = test_rows.head(60).copy()\n        seed_labels = {int(r[\"image_id\"]): f\"seed_{species}_{int(r['image_id'])}\" for r in test_rows.to_dict(\"records\")}\n        print(f\"\\n[{species}] extracting test features: {len(test_rows)}\")\n        test_items = [\n            feature_item(row, species, seed_labels, detector_name, detector, args.max_side, args.img_size)\n            for row in test_rows.to_dict(\"records\")\n        ]\n        item_cache[species] = test_items\n        feature_rows.extend(\n            {\n                \"species\": it.species,\n                \"split\": \"test\",\n                \"image_id\": it.image_id,\n                \"orientation\": it.orientation,\n                \"body_kind\": it.body_kind,\n                \"low_light\": bool(it.low_light),\n                \"n_keypoints\": int(len(it.keypoints)),\n                \"quality\": float(it.quality),\n            }\n            for it in test_items\n        )\n        test_pairs = cosine_pairs(test_items, args.top_k, args.pair_budget_per_species)\n        test_scores = score_pairs(test_items, test_pairs, detector_name)\n        test_scores.to_csv(reports_dir / f\"{VERSION}_{species}_test_pair_scores.csv\", index=False)\n        test_pair_scores[species] = test_scores\n\n        train_rows = sample_train_rows(view_df[view_df[\"species_id\"].eq(species)].copy(), args.train_images_per_species)\n        print(f\"[{species}] extracting train calibration features: {len(train_rows)}\")\n        train_labels = {int(r[\"image_id\"]): str(r.get(\"identity\", \"\")) for r in train_rows.to_dict(\"records\")}\n        train_items = [\n            feature_item(row, species, train_labels, detector_name, detector, args.max_side, args.img_size)\n            for row in train_rows.to_dict(\"records\")\n        ]\n        feature_rows.extend(\n            {\n                \"species\": it.species,\n                \"split\": \"train_calibration\",\n                \"image_id\": it.image_id,\n                \"orientation\": it.orientation,\n                \"body_kind\": it.body_kind,\n                \"low_light\": bool(it.low_light),\n                \"n_keypoints\": int(len(it.keypoints)),\n                \"quality\": float(it.quality),\n            }\n            for it in train_items\n        )\n        train_pairs = calibration_pairs(train_items, args.top_k, args.train_pairs_per_species)\n        cal_scores = score_pairs(train_items, train_pairs, detector_name)\n        cal_scores.to_csv(reports_dir / f\"{VERSION}_{species}_train_calibration_pair_scores.csv\", index=False)\n        cal_thresholds[species] = {}\n        for profile, cfg in PROFILE.items():\n            target = cfg[\"precision\"][species]\n            cal_thresholds[species][profile] = threshold_for_precision(cal_scores, target, species)\n            cal_thresholds[species][profile][\"threshold\"] *= float(cfg[\"thr_scale\"])\n            cal_thresholds[species][profile][\"profile\"] = profile\n            cal_thresholds[species][profile][\"species\"] = species\n        source_rows.extend(cal_thresholds[species].values())\n\n    pd.DataFrame(source_rows).to_csv(reports_dir / f\"{VERSION}_calibrated_thresholds.csv\", index=False)\n    pd.DataFrame(feature_rows).to_csv(reports_dir / f\"{VERSION}_feature_diagnostics.csv\", index=False)\n\n    texas_rows = test_rows_by_species[TEXAS].copy()\n    if args.smoke:\n        texas_rows = texas_rows.head(60).copy()\n    texas_seed_labels = {int(r[\"image_id\"]): f\"seed_{TEXAS}_{int(r['image_id'])}\" for r in texas_rows.to_dict(\"records\")}\n    print(\"[Texas] independent belly-dot graph; no past submission seed labels\")\n    texas_variants = texas_labels_from_views(texas_rows, texas_seed_labels, args, reports_dir, viz_dir)\n\n    variants: dict[str, dict] = {\n        \"independent_swing\": {\n            \"profiles\": {\n                \"LynxID2025\": \"independent_swing\",\n                \"SalamanderID2025\": \"independent_swing\",\n                \"SeaTurtleID2022\": \"independent_balanced\",\n            },\n            \"texas\": \"splitmerge_swing\",\n        },\n        \"independent_wild\": {\n            \"profiles\": {\n                \"LynxID2025\": \"independent_wild\",\n                \"SalamanderID2025\": \"independent_wild\",\n                \"SeaTurtleID2022\": \"independent_swing\",\n            },\n            \"texas\": \"splitmerge_swing\",\n        },\n    }\n\n    candidate_rows: list[dict] = []\n    accepted_all: list[dict] = []\n    written: dict[str, str] = {}\n    for variant, spec in variants.items():\n        updates: dict[int, str] = {}\n        for species in REID_SPECIES:\n            rows = test_rows_by_species[species]\n            if args.smoke:\n                rows = rows.head(60).copy()\n            seed_labels = {int(r[\"image_id\"]): f\"seed_{species}_{int(r['image_id'])}\" for r in rows.to_dict(\"records\")}\n            profile = spec[\"profiles\"].get(species)\n            if profile is None:\n                updates.update({image_id: f\"cluster_{species}_singleton_{image_id}\" for image_id in seed_labels})\n                continue\n            threshold = float(cal_thresholds[species][profile][\"threshold\"])\n            cfg = PROFILE[profile]\n            labels, accepted = graph_merge_species(\n                species,\n                rows,\n                seed_labels,\n                test_pair_scores.get(species, pd.DataFrame()),\n                threshold=threshold,\n                min_inliers=int(cfg[\"min_inliers\"][species]),\n                max_edges=int(cfg[\"max_edges\"][species]),\n                variant=variant,\n                preserve_existing=False,\n            )\n            updates.update(labels)\n            accepted_all.extend(accepted)\n        texas_choice = spec.get(\"texas\")\n        if texas_choice and texas_choice in texas_variants:\n            updates.update(texas_variants[texas_choice])\n        elif not texas_rows.empty:\n            updates.update({image_id: f\"cluster_{TEXAS}_singleton_{image_id}\" for image_id in texas_seed_labels})\n        out_path = submissions_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub = build_submission(sample, view_df, updates, variant, out_path)\n        write_species_fragments(sub, view_df, variant, fragments_dir)\n        written[variant] = str(out_path)\n        candidate_rows.extend(summarize_submission(sub, reference, view_df, variant))\n        print(f\"wrote {out_path}\")\n\n    candidate_report = pd.DataFrame(candidate_rows)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n    shape_score_report = candidate_shape_scores(candidate_report)\n    shape_score_report.to_csv(reports_dir / f\"{VERSION}_candidate_shape_scores.csv\", index=False)\n    accepted_report = pd.DataFrame(accepted_all)\n    accepted_report.to_csv(reports_dir / f\"{VERSION}_accepted_edges.csv\", index=False)\n    if viz_dir is not None and not accepted_report.empty:\n        save_edge_preview(view_df, accepted_report, viz_dir / f\"{VERSION}_accepted_edge_preview.jpg\", args.visual_limit)\n\n    # Convenience output: pick the candidate whose per-species cluster shape is\n    # closest to a sane non-collapsed regime. This avoids both giant-component\n    # failure and the quieter failure where a submission is mostly singleton\n    # clusters.\n    recommended_variant, recommended, shape_score_report = choose_recommended_submission(candidate_report, written)\n    shape_score_report.to_csv(reports_dir / f\"{VERSION}_candidate_shape_scores.csv\", index=False)\n    if recommended.exists():\n        recommended_df = pd.read_csv(recommended)\n        recommended_df.to_csv(output_root / \"submission.csv\", index=False)\n        recommended_df.to_csv(Path.cwd() / \"submission.csv\", index=False)\n\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"view_manifest\": str(view_manifest),\n        \"reference_submission_optional\": str(reference_path) if reference_path else None,\n        \"detector\": detector_name,\n        \"written_submissions\": written,\n        \"recommended_variant\": recommended_variant,\n        \"recommended_first_review\": str(recommended),\n        \"kaggle_top_level_submission\": str(Path.cwd() / \"submission.csv\"),\n        \"reports\": {\n            \"candidate_report\": str(reports_dir / f\"{VERSION}_candidate_report.csv\"),\n            \"candidate_shape_scores\": str(reports_dir / f\"{VERSION}_candidate_shape_scores.csv\"),\n            \"thresholds\": str(reports_dir / f\"{VERSION}_calibrated_thresholds.csv\"),\n            \"feature_diagnostics\": str(reports_dir / f\"{VERSION}_feature_diagnostics.csv\"),\n            \"accepted_edges\": str(reports_dir / f\"{VERSION}_accepted_edges.csv\"),\n            \"texas_template_report\": str(reports_dir / f\"{VERSION}_Texas_template_report.csv\"),\n        },\n        \"species_fragments\": str(fragments_dir),\n        \"notes\": [\n            \"Main candidates are independent clusters built from singleton seeds.\",\n            \"No previous submission is auto-discovered or used as a clustering base.\",\n            f\"Global image prep pads to a species-specific square background and resizes to {args.img_size}x{args.img_size}: Lynx uses black padding; Salamander, SeaTurtle, and Texas use neutral-grey padding.\",\n            \"Lynx uses the original image as signal source; the SAM fused mask only gates subject-only CLAHE/brightness adjustment.\",\n            \"Salamander uses SAM-clean output with neutral-grey background and strictly no rotation or flip preprocessing.\",\n            \"SeaTurtle keeps the existing clean-view route, with only the global square-pad/resize normalization added.\",\n            \"Texas uses SAM+YOLO final/union output with neutral-grey background, no oval crop, and no vertical flip.\",\n            \"ReID graph merges now require mutual-rank agreement, per-image degree caps, and per-species component caps.\",\n            \"Lynx/Salamander/SeaTurtle component caps are train-prior tiers: p95, p99, and observed train max.\",\n            \"submission.csv is copied from the candidate with the best species-shape sanity score, not merely the first non-collapsed file.\",\n            \"Texas orientation rule is head_already_top_no_flip; neither template creation nor pair scoring uses vertical flipping.\",\n            \"Per-species fragment CSVs are written so expensive species notebooks can be merged later.\",\n        ],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    print(\"\\nCandidate report:\")\n    print(candidate_report.to_string(index=False))\n    print(\"\\nCandidate shape scores:\")\n    print(shape_score_report.to_string(index=False))\n    if not accepted_report.empty:\n        print(\"\\nAccepted edge counts:\")\n        print(accepted_report.groupby([\"variant\", \"species\"]).size().reset_index(name=\"n_edges\").to_string(index=False))\n    print(f\"\\nRecommended first review/upload candidate ({recommended_variant}): {recommended}\")\n    print(f\"Convenience submission.csv: {output_root / 'submission.csv'}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
print("scripts written")
            

In [ ]:
import importlib.util
import subprocess
import sys

packages = []
if importlib.util.find_spec("cv2") is None:
    packages.append("opencv-python-headless")

if packages:
    print("installing:", packages)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
else:
    print("dependencies available")
            

In [ ]:
from pathlib import Path
import subprocess
import sys

OUTPUT_ROOT = Path("/kaggle/working/animalclef_sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427")

cmd = [
    sys.executable,
    "sam_yolo_species_submission_v20260427.py",
    "--output-root", str(OUTPUT_ROOT),
    "--top-k", "48",
    "--pair-budget-per-species", "85000",
    "--train-images-per-species", "850",
    "--train-pairs-per-species", "3200",
    "--texas-pair-budget", "18000",
    "--save-visualizations",
]

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
            

In [ ]:
from pathlib import Path
import json
import pandas as pd

out = Path("/kaggle/working/animalclef_sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427")
reports = out / "reports"
subs = out / "submissions"

display(pd.read_csv(reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_calibrated_thresholds.csv"))
display(pd.read_csv(reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_candidate_report.csv"))
shape_scores = reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_candidate_shape_scores.csv"
if shape_scores.exists():
    display(pd.read_csv(shape_scores))
feature_diag = reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_feature_diagnostics.csv"
if feature_diag.exists():
    fd = pd.read_csv(feature_diag)
    display(fd.groupby(["species", "split", "orientation", "body_kind", "low_light"]).size().reset_index(name="n").head(80))
texas_template = reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_Texas_template_report.csv"
if texas_template.exists():
    display(pd.read_csv(texas_template).head(100))
accepted = reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_accepted_edges.csv"
if accepted.exists():
    df = pd.read_csv(accepted)
    display(df.head(100))
    if len(df):
        display(df.groupby(["variant", "species"]).size().reset_index(name="n_edges"))

summary = json.loads((reports / "sam_yolo_speciesfusion_shapeaware_shortlabels_v20260427_summary.json").read_text())
print(json.dumps(summary, indent=2)[:5000])

print("\nSubmission files:")
for p in sorted(subs.glob("*.csv")):
    print(p)
print("\nSpecies fragment files:")
for p in sorted((out / "species_fragments").glob("*.csv")):
    print(p)
print("\nConvenience submission:")
print(out / "submission.csv")
            